# 10 — LaTeX parsing (Member 2)

Phạm vi: find_root → gather → build_tree cho từng version.
Root có documentclass trong preamble được include; hỗ trợ tên file có dấu chấm,
input/include/subfile/import/subimport và các alias tương ứng.

Mục đích bản này: push GitHub để các thành viên xem code và kiểm tra.
Không yêu cầu hoàn thành toàn pipeline hoặc benchmark corpus trước khi push review.
Các hàm chuẩn hóa/tách câu chỉ được đọc nguyên từ notebook 11 của Member 4;
notebook 10 không chứa phương án A/B đã sửa và không ghi lại notebook 11.


## 1. Kết nối dữ liệu

Nếu `arxiv-project` đang ở **Được chia sẻ với tôi**, thêm lối tắt của thư mục
vào **Drive của tôi** trước. Đăng nhập đúng tài khoản có quyền truy cập.
Nếu lối tắt có tên hoặc vị trí khác, sửa `RAW_DIR` trong cell cấu hình.
Khi chạy local, cell mount sẽ được bỏ qua; hãy đặt `RAW_DIR` thành đường dẫn local.

In [ ]:
try:
    from google.colab import drive
except ImportError:
    print("Đang chạy ngoài Colab: bỏ qua mount Drive, cấu hình RAW_DIR local ở cell sau.")
else:
    drive.mount("/content/drive")

## 2. Chọn một version

Chỉ cần chỉnh ba giá trị dưới đây. `source_dir` phải trỏ trực tiếp vào thư mục
version chứa `.tex`, không trỏ vào toàn bộ `raw` hoặc paper chứa nhiều version.
Nếu không tìm thấy đường dẫn, dùng bảng **Files** của Colab để lấy **Copy path**.

In [ ]:
from pathlib import Path

RAW_DIR = Path("/content/drive/MyDrive/arxiv-project/data/raw")
PAPER_ID = "2609-37833"
VERSION = "v1"

source_dir = RAW_DIR / PAPER_ID / VERSION

if not source_dir.is_dir():
    raise FileNotFoundError(
        f"Không tìm thấy thư mục version: {source_dir}\n"
        "Kiểm tra mount Drive, lối tắt arxiv-project và giá trị RAW_DIR."
    )

print("Source directory:", source_dir)
print("\nNội dung trực tiếp (tối đa 30 mục):")
entries = sorted(source_dir.iterdir(), key=lambda path: path.name)
for path in entries[:30]:
    kind = "DIR" if path.is_dir() else "FILE"
    print(f"[{kind}] {path.name}")

tex_files = sorted(
    path for path in source_dir.rglob("*")
    if path.is_file() and path.suffix.lower() == ".tex"
)
print("\nSố file .tex:", len(tex_files))
for path in tex_files[:20]:
    print("-", path.relative_to(source_dir).as_posix())
if not tex_files:
    print("Chưa thấy .tex: kiểm tra thư mục, file nén hoặc version không có TeX.")

## 3. Hàm hỗ trợ

Mask comment và các môi trường verbatim phổ biến; đọc file với encoding fallback;
dựng graph `input/include` sơ bộ để đánh giá ứng viên root. Cell này chỉ định nghĩa hàm.

In [ ]:
from pathlib import Path
import re


_VERBATIM_ENVS = {
    "verbatim",
    "verbatim*",
    "Verbatim",
    "lstlisting",
    "minted",
    "comment",
}

_DOCUMENTCLASS_RE = re.compile(
    r"\\documentclass\b"
    r"\s*(?:\[[^\]]*\]\s*)?"
    r"\{([^{}]+)\}",
    flags=re.DOTALL,
)

_BEGIN_DOCUMENT_RE = re.compile(
    r"\\begin\s*\{\s*document\s*\}"
)

_END_DOCUMENT_RE = re.compile(
    r"\\end\s*\{\s*document\s*\}"
)

_SOURCE_ACTIVE_CHAR_RE = re.compile(r"[%\\]")
_SOURCE_VERB_RE = re.compile(r"\\verb\*?(?![A-Za-z])")
_SOURCE_BEGIN_ENV_RE = re.compile(r"\\begin\s*\{([^{}]+)\}")
_SOURCE_CONTROL_WORD_RE = re.compile(r"\\[A-Za-z@]+")
_SOURCE_BLANK_RE = re.compile(r"[^\r\n]")


def _mask_non_code(text):
    # Spans to blank (increasing, disjoint), applied once at the end: blanking
    # character by character was the slowest step of the parser.
    spans = []

    def blank(start, end):
        spans.append((start, end))

    i = 0
    n = len(text)

    while i < n:
        marker = _SOURCE_ACTIVE_CHAR_RE.search(text, i)
        if marker is None:
            break
        i = marker.start()
        if text[i] == "%":
            backslashes = 0
            j = i - 1

            while j >= 0 and text[j] == "\\":
                backslashes += 1
                j -= 1

            if backslashes % 2 == 0:
                end = text.find("\n", i)

                if end == -1:
                    end = n

                blank(i, end)
                i = end
                continue

        if text[i] == "\\":
            if i + 1 < n and not text[i + 1].isalpha():
                i += 2
                continue

            verb_match = _SOURCE_VERB_RE.match(text, i) if text.startswith("\\verb", i) else None

            if verb_match:
                delimiter_pos = verb_match.end()

                if (
                    delimiter_pos < n
                    and not text[delimiter_pos].isspace()
                ):
                    delimiter = text[delimiter_pos]
                    end = text.find(delimiter, delimiter_pos + 1)
                    newline = text.find("\n", delimiter_pos + 1)

                    if end != -1 and (newline == -1 or end < newline):
                        blank(i, end + 1)
                        i = end + 1
                        continue

            # Bỏ qua các môi trường chứa literal text.
            env_match = _SOURCE_BEGIN_ENV_RE.match(text, i) if text.startswith("\\begin", i) else None

            if env_match and env_match.group(1) in _VERBATIM_ENVS:
                env_name = env_match.group(1)
                end_pattern = re.compile(
                    r"\\end\s*\{" + re.escape(env_name) + r"\}"
                )

                end_match = end_pattern.search(
                    text,
                    env_match.end(),
                )

                end = end_match.end() if end_match else n
                blank(i, end)
                i = end
                continue

            # The rest of the command name holds no % or backslash:
            # the next search skips it.
            i += 1
            continue

        i += 1

    if not spans:
        return text
    pieces, pos = [], 0
    for start, end in spans:
        pieces.append(text[pos:start])
        pieces.append(_SOURCE_BLANK_RE.sub(" ", text[start:end]))
        pos = end
    pieces.append(text[pos:])
    return "".join(pieces)


def _brace_table(text, token_re):
    """{position of "{": position after its "}"} of every closed group of text, in one pass.

    token_re matches a command (skipped whole, as the group scanners skip it) or a brace.
    Used once a text has shown an unclosed group: then the scan from every later "{" would
    also run to the end of the text, quadratic on a source with thousands of stray braces.
    """
    table, stack = {}, []
    for token in token_re.finditer(text):
        if token.group() == "{":
            stack.append(token.start())
        elif token.group() == "}" and stack:
            table[stack.pop()] = token.end()
    return table


def _read_tex(path):
    raw = path.read_bytes()

    try:
        return raw.decode("utf-8-sig"), None
    except UnicodeDecodeError:
        return (
            raw.decode("latin-1"),
            "UTF-8 decoding failed; used Latin-1 fallback.",
        )


def _strip_tex_comments(text):
    """Remove percent comments and comment environments, preserving verbatim."""
    # Text is kept in whole runs (text[copied:...]); only a comment or a comment
    # environment ends a run. Commands, verbatim and \verb are kept as they are.
    pieces = []
    copied = i = 0
    n = len(text)
    while i < n:
        marker = _SOURCE_ACTIVE_CHAR_RE.search(text, i)
        if marker is None:
            break
        i = marker.start()
        if text[i] == "%":
            end = text.find("\n", i)
            pieces.append(text[copied:i])
            i = copied = n if end == -1 else end + 1   # the comment and its newline
            continue
        # Control symbols, including \%, \\, \{.
        if i + 1 < n and not text[i + 1].isalpha():
            i += 2
            continue
        verb = _SOURCE_VERB_RE.match(text, i) if text.startswith("\\verb", i) else None
        if verb:
            start = verb.end()
            if start < n and not text[start].isspace():
                end = text.find(text[start], start + 1)
                newline = text.find("\n", start + 1)
                if end != -1 and (newline == -1 or end < newline):
                    i = end + 1
                    continue
        env = _SOURCE_BEGIN_ENV_RE.match(text, i) if text.startswith("\\begin", i) else None
        if env and env.group(1) in _VERBATIM_ENVS:
            closing = re.compile(
                r"\\end\s*\{" + re.escape(env.group(1)) + r"\}"
            ).search(text, env.end())
            end = closing.end() if closing else n
            if env.group(1) == "comment":
                pieces.append(text[copied:i])
                if "\n" in text[i:end]:
                    pieces.append("\n")
                copied = end
            i = end
            continue
        i += 1   # a control word: its letters hold no marker
    pieces.append(text[copied:])
    return "".join(pieces)


## 4. Tìm entry point biên dịch

Root là file khởi chạy tài liệu, không nhất thiết là file trực tiếp chứa documentclass.
Hàm xét documentclass reachable qua include; trả documentclass_source để phân biệt preamble với root.
Các hàm dò/resolve source bên dưới được dùng chung cho find_root và gather.
Hỗ trợ input/include/subfile/import/subimport và các alias inputfrom/includefrom.
Tên có dấu chấm vẫn được thử với đuôi .tex, ví dụ Sections/1.Introduction.tex.
Lệnh nằm trong ngoặc vẫn được theo (`\resizebox{..}{!}{\input{table}}`,
`\IfFileExists{f}{\input{f}}{}`); chỉ bỏ qua thân các định nghĩa (`\newcommand`, `\def`,
`\newenvironment`, `\NewDocumentCommand`...), vì macro không được gọi thì không được biên dịch.


In [ ]:
# find_root and gather share the same lexical scan and path resolution rules.
import bisect

_SOURCE_COMMAND_OR_GROUP_RE = re.compile(r"[{\\]")
_SOURCE_COMMAND = re.compile(r"\\([A-Za-z@]+\*?|[^\r\n])")
_SOURCE_SIMPLE = {"input", "include", "subfile", "subfileinclude"}
_SOURCE_IMPORT = {"import", "subimport", "inputfrom", "subinputfrom", "includefrom", "subincludefrom"}
_SUBFILE_COMMANDS = {"subfile", "subfileinclude"}
_SOURCE_EXTENSIONS = {".tex", ".sty", ".cls", ".def", ".ltx", ".latex", ".bbl"}
_SOURCE_TOKEN_RE = re.compile(
    r"\\(input|include|subfile|subfileinclude|import|subimport|inputfrom|subinputfrom|includefrom"
    r"|subincludefrom|endinput|documentclass|begin|end)(?![A-Za-z@])\*?")
_SOURCE_DEFINITION_RE = re.compile(
    r"\\(?:(?P<def>[gex]?def)|(?P<macro>(?:re)?newcommand|providecommand|DeclareRobustCommand"
    r"|DeclareMathOperator)|(?P<doc>(?:New|Renew|Provide|Declare)DocumentCommand)"
    r"|(?P<env>(?:re)?newenvironment)|(?P<docenv>(?:New|Renew|Provide|Declare)DocumentEnvironment))"
    r"(?![A-Za-z@])\*?")
_SOURCE_DEF_PARAMETERS = re.compile(r"(?:#\d|[^{}\\\n%])*\{")   # \def\x#1#2{ or \def\x#1.{


_SOURCE_GROUP_STOP_RE = re.compile(r"[\\{}]")
_SOURCE_BRACE_TOKEN_RE = re.compile(r"\\(?:[A-Za-z@]+\*?|[^\r\n])|[{}]")
_SOURCE_UNCLOSED = {}   # text with an unclosed group -> _brace_table of it


def _source_group(text, pos):
    if pos >= len(text) or text[pos] != "{":
        return None
    table = _SOURCE_UNCLOSED.get(text)
    if table is not None:   # see the end of this function
        end = table.get(pos)
        return (text[pos + 1:end - 1], end) if end is not None else None
    start, depth = pos, 0
    while pos < len(text):
        marker = _SOURCE_GROUP_STOP_RE.search(text, pos)   # other characters change nothing
        if marker is None:
            break
        pos = marker.start()
        if text[pos] == "\\":
            token = _SOURCE_COMMAND.match(text, pos)
            if token:
                pos = token.end()
                continue
        if text[pos] == "{":
            depth += 1
        elif text[pos] == "}":
            depth -= 1
            if depth == 0:
                return text[start + 1:pos], pos + 1
        pos += 1
    # Unclosed: this scan ran to the end of the text, as would the scan from every later "{".
    if len(_SOURCE_UNCLOSED) > 4:   # a few texts at most: they can be large
        _SOURCE_UNCLOSED.clear()
    _SOURCE_UNCLOSED[text] = _brace_table(text, _SOURCE_BRACE_TOKEN_RE)
    return None


def _source_escaped(masked, pos):
    """True when the backslash at pos is the second half of an escaped one (\\\\input)."""
    count = 0
    while pos > 0 and masked[pos - 1] == "\\":
        count += 1
        pos -= 1
    return count % 2 == 1


def _source_skip_space(masked, pos):
    while pos < len(masked) and masked[pos].isspace():
        pos += 1
    return pos


def _source_definition_spans(masked):
    """Disjoint (start, end) of macro/environment definitions: their bodies are not compiled."""
    spans = []
    for match in _SOURCE_DEFINITION_RE.finditer(masked):
        start = match.start()
        if (spans and start < spans[-1][1]) or _source_escaped(masked, start):
            continue   # nested in a definition already skipped, or \\def (line break + text)
        pos = _source_skip_space(masked, match.end())
        if match.group("def"):
            name = _SOURCE_COMMAND.match(masked, pos)
            parameters = _SOURCE_DEF_PARAMETERS.match(masked, name.end()) if name else None
            group = _source_group(masked, parameters.end() - 1) if parameters else None
            if group:
                spans.append((start, group[1]))
            continue
        if masked.startswith("{", pos):   # {\name} or {envname}
            group = _source_group(masked, pos)
            if not group:
                continue
            pos = group[1]
        else:                              # \name
            name = _SOURCE_COMMAND.match(masked, pos)
            if not name:
                continue
            pos = name.end()
        # body: 1 group; \newenvironment, \NewDocumentCommand: 2; \NewDocumentEnvironment: 3
        groups = 3 if match.group("docenv") else 2 if match.group("env") or match.group("doc") else 1
        end = None
        while groups:
            pos = _source_skip_space(masked, pos)
            if masked.startswith("[", pos):   # [n], [default]
                close = masked.find("]", pos)
                if close < 0:
                    break
                pos = close + 1
                continue
            group = _source_group(masked, pos)
            if not group:
                break
            pos = end = group[1]
            groups -= 1
        if end is not None:
            spans.append((start, end))
    return spans


def _source_tokens(text):
    """Return source commands up to the endinput line, with original offsets.

    Commands inside braces count (\\resizebox{..}{!}{\\input{t}}, \\IfFileExists{f}{\\input{f}}{});
    only definition bodies are skipped, as a macro that is never used is never compiled.
    """
    masked = _mask_non_code(text)
    skipped = _source_definition_spans(masked)
    skipped_starts = [span[0] for span in skipped]
    tokens = []
    pos, stop = 0, len(text)
    while pos < stop:
        token = _SOURCE_TOKEN_RE.search(masked, pos, stop)
        if token is None:
            break
        start, pos = token.start(), token.end()
        inside = bisect.bisect_right(skipped_starts, start) - 1
        if _source_escaped(masked, start) or (inside >= 0 and start < skipped[inside][1]):
            continue
        command = token.group(1)
        if command == "endinput":
            newline = text.find("\n", pos, stop)
            if newline != -1:
                stop = newline + 1
            tokens.append({"command": command, "start": start, "end": pos})
            continue
        if command in {"documentclass", "begin", "end"}:
            pattern = {"documentclass": _DOCUMENTCLASS_RE, "begin": _BEGIN_DOCUMENT_RE,
                       "end": _END_DOCUMENT_RE}[command]
            marker = pattern.match(masked, start, stop)
            if marker:
                pos = marker.end()
                tokens.append({"command": command if command == "documentclass" else command + "_document",
                               "start": start, "end": pos,
                               "argument": marker.group(1).strip() if command == "documentclass" else ""})
            continue
        if command not in _SOURCE_SIMPLE | _SOURCE_IMPORT:
            continue

        expected = 2 if command in _SOURCE_IMPORT else 1
        cursor = pos
        args = []
        for _ in range(expected):
            while cursor < stop and masked[cursor].isspace():
                cursor += 1
            group = _source_group(masked, cursor)
            if group and group[1] <= stop:
                args.append(_strip_tex_comments(text[cursor + 1:group[1] - 1]).strip())
                cursor = group[1]
            elif command == "input" and not args:
                bare = re.match(r'"([^"\n]+)"|([^\s{}\\%]+)', masked[cursor:stop])
                if bare:
                    args.append(bare.group(1) or bare.group(2))
                    cursor += bare.end()
            else:
                break
        pos = max(cursor, pos)
        tokens.append({"command": command, "start": start, "end": pos,
                       "argument": args[-1] if len(args) == expected else "",
                       "directory": args[0] if expected == 2 and len(args) == 2 else None,
                       "malformed": len(args) != expected})
    return tokens, stop


def _source_window(tokens, stop, subfile_mode):
    """A subfile contributes its document body, or the entire file if it is a fragment."""
    if not subfile_mode:
        return 0, stop, False
    begin = next((i for i, token in enumerate(tokens) if token["command"] == "begin_document"), None)
    has_class = any(token["command"] == "documentclass" for token in tokens)
    if begin is None:
        return 0, stop, has_class
    end = next((token["start"] for token in tokens[begin + 1:] if token["command"] == "end_document"), None)
    return tokens[begin]["end"], stop if end is None else end, end is None


def resolve_source_include(spec, current, source_dir, working_dir, import_dir=None):
    """Use the import context, compilation directory, then the caller directory."""
    argument = spec["argument"]
    directory = spec["directory"]
    failure = {"targets": [], "error": "unsupported_path", "import_dir": import_dir}
    if spec["malformed"] or not argument or any(c in argument + (directory or "") for c in "\\#{}\x00"):
        return failure
    path = Path(argument)
    if path.is_absolute() or path.drive:
        return {**failure, "error": "outside_source"}
    variants = [path] if path.suffix.lower() in _SOURCE_EXTENSIONS else [Path(str(path) + ".tex"), path]
    next_import = import_dir
    if spec["command"] in _SOURCE_IMPORT:
        d = Path(directory or ".")
        if d.is_absolute() or d.drive:
            return {**failure, "error": "outside_source"}
        anchor = (import_dir or working_dir) if spec["command"].startswith("sub") else working_dir
        next_import = (anchor / d).resolve()
        bases = [next_import]
    else:
        bases = ([import_dir] if import_dir is not None else []) + [working_dir, current.parent]
    matches = []
    outside = False
    for base in dict.fromkeys(bases):
        for variant in variants:
            target = (base / variant).resolve()
            if not target.is_relative_to(source_dir):
                outside = True
            elif target not in matches and target.is_file():
                matches.append(target)
    if matches and spec["command"] in _SUBFILE_COMMANDS:
        next_import = matches[0].parent
    return {"targets": matches, "import_dir": next_import,
            "error": "outside_source" if outside else "missing_file"}


def find_root(source_dir):
    """Choose a compilation entry using the preamble and includes in execution order."""
    source_dir = Path(source_dir).expanduser().resolve()
    if not source_dir.is_dir():
        raise NotADirectoryError(source_dir)
    result = {"status": "no_root", "source_dir": str(source_dir), "root_path": None,
              "selection_reason": [], "ambiguous": False, "candidates": [],
              "tex_file_count": 0, "warnings": []}
    entries = sorted({path.resolve() for path in source_dir.rglob("*")
                      if path.suffix.lower() == ".tex" and path.is_file()
                      and path.resolve().is_relative_to(source_dir)})
    result["tex_file_count"] = len(entries)
    scanned, resolutions = {}, {}
    incoming = {path: set() for path in entries}
    reachable_by_entry = {}
    rel = lambda path: path.relative_to(source_dir).as_posix()

    def scan(path):
        if path not in scanned:
            try:
                raw, warning = _read_tex(path)
                scanned[path] = _source_tokens(raw)
                if warning:
                    result["warnings"].append(f"{rel(path)}: {warning}")
            except OSError as exc:
                result["warnings"].append(f"{rel(path)}: {exc}")
                scanned[path] = ([], 0)
        return scanned[path]

    for entry in entries:
        declarations, edges = [], set()
        visited, completed = set(), set()
        begin_file = None
        ended = False

        def walk(path, stack, import_dir=None, subfile_mode=False):
            nonlocal begin_file, ended
            key = (path, import_dir, subfile_mode)
            if path in stack or key in completed or len(stack) >= 64 or len(completed) >= 5000:
                return
            completed.add(key)
            visited.add(path)
            tokens, stop = scan(path)
            start, stop, _ = _source_window(tokens, stop, subfile_mode)
            for token in tokens:
                if ended:
                    return
                if not start <= token["start"] < stop:
                    continue
                command = token["command"]
                if command == "documentclass" and begin_file is None:
                    declarations.append((path, token["argument"]))
                elif command == "begin_document" and begin_file is None:
                    begin_file = path
                elif command == "end_document":
                    ended = True
                elif command in _SOURCE_SIMPLE | _SOURCE_IMPORT:
                    key = (path, token["start"], entry.parent, import_dir)
                    if key not in resolutions:
                        resolutions[key] = resolve_source_include(token, path, source_dir, entry.parent, import_dir)
                    resolution = resolutions[key]
                    if resolution["targets"]:
                        target = resolution["targets"][0]
                        edges.add((path, target))
                        walk(target, (*stack, path), resolution["import_dir"], command in _SUBFILE_COMMANDS)

        walk(entry, ())
        if not declarations:
            continue
        class_source, class_name = declarations[0]
        class_type = class_name.casefold()
        reachable_by_entry[entry] = visited
        if begin_file is not None and class_type not in {"standalone", "subfiles"}:
            for parent, target in edges:
                if parent != target and target in incoming:
                    incoming[target].add(parent)
        score, reasons = 0, []
        for condition, points, reason in [
            (begin_file == entry, 60, "document body starts in this entry file"),
            (begin_file is not None and begin_file != entry, 20, "document body reachable via includes"),
            (ended, 5, "end{document} reachable"),
            (bool(set(re.split(r"[^a-z0-9]+", entry.stem.casefold())) & {"main", "ms"}), 20, "main/ms filename"),
            (bool(set(re.split(r"[^a-z0-9]+", entry.stem.casefold())) & {"rebuttal", "response", "supp", "supplement", "supplementary"}), -40, "auxiliary document filename"),
        ]:
            if condition:
                score += points
                reasons.append(f"{points:+d}: {reason}")
        excluded = class_type in {"standalone", "subfiles"}
        if excluded:
            reasons.append(f"Not a standalone compilation entry for the paper: class {class_name}")
        result["candidates"].append({
            "path": rel(entry), "documentclass": class_name, "documentclass_source": rel(class_source),
            "excluded": excluded, "score": score + min(len(visited) - 1, 5),
            "has_begin_document": begin_file == entry,
            "has_reachable_begin_document": begin_file is not None,
            "reachable_tex_count": sum(path.suffix.lower() == ".tex" for path in visited),
            "included_by": [], "reasons": reasons,
        })

    for candidate in result["candidates"]:
        path = source_dir / candidate["path"]
        parents = incoming[path]
        candidate["included_by"] = sorted(rel(parent) for parent in parents)
        if parents - reachable_by_entry[path]:
            candidate["excluded"] = True
            candidate["reasons"].append("Included by another compilation entry")
        elif not parents:
            candidate["score"] += 30
            candidate["reasons"].append("+30: not included by another source")

    result["candidates"].sort(key=lambda c: (c["excluded"], not c["has_reachable_begin_document"],
                                             -c["score"], c["path"]))
    eligible = [c for c in result["candidates"] if not c["excluded"]]
    if not eligible:
        result["warnings"].append("No usable documentclass reachable from any entry.")
        return result
    chosen = eligible[0]
    result.update(status="ok", root_path=chosen["path"], selection_reason=chosen["reasons"])
    if len(eligible) > 1 and eligible[1]["has_reachable_begin_document"] == chosen["has_reachable_begin_document"] and chosen["score"] - eligible[1]["score"] < 15:
        result["ambiguous"] = True
        result["warnings"].append("Multiple plausible compilation entries; inspect candidates.")
    if not chosen["has_reachable_begin_document"]:
        result["ambiguous"] = True
        result["warnings"].append("Selected entry has no reachable begin{document}.")
    return result


## 5. Chạy và xem ứng viên

Với version trong ảnh, kỳ vọng chọn `main.tex` nếu khai báo documentclass hợp lệ.
Bảng bên dưới giữ cả ứng viên bị loại để tiện kiểm tra.

In [ ]:
from IPython.display import display

root_result = find_root(source_dir)

print("Paper:", PAPER_ID, "| Version:", VERSION)
print("Status:", root_result["status"])
print("Root:", root_result["root_path"])
print("Ambiguous:", root_result["ambiguous"])
print("Number of .tex files:", root_result["tex_file_count"])

print("\nSelection reasons:")
for reason in root_result["selection_reason"]:
    print("-", reason)

print("\nWarnings:")
if not root_result["warnings"]:
    print("(none)")
for warning in root_result["warnings"]:
    print("-", warning)

rows = [
    {key: item[key] for key in (
        "path", "documentclass", "excluded", "score",
        "has_reachable_begin_document", "reachable_tex_count", "included_by"
    )}
    for item in root_result["candidates"]
]
if rows:
    try:
        import pandas as pd
    except ImportError:
        for row in rows:
            print(row)
    else:
        display(pd.DataFrame(rows))

## 6. Kiểm tra root được chọn

Xem khai báo documentclass và phần đầu source để xác nhận đây là bài chính.
Không có root thì cell này chỉ thông báo, không đọc một đường dẫn rỗng.

In [ ]:
root_file = None
if root_result["status"] != "ok":
    print("Chưa chọn được root. Xem warnings.")
else:
    root_file = Path(root_result["source_dir"]) / root_result["root_path"]
    selected = next(c for c in root_result["candidates"] if c["path"] == root_result["root_path"])
    print("Compilation entry:", root_result["root_path"])
    print("Documentclass found in:", selected["documentclass_source"])
    root_text, _ = _read_tex(root_file)
    print("\n--- Entry preview ---")
    print(root_text[:4000])


## Sau Task 1

Giữ `source_dir`, `root_file` và `root_result` để chạy Task 2 bên dưới.
Nếu root mơ hồ, kiểm tra và chọn lại root trước khi gather.


# Task 2 — Gom source LaTeX nhiều file

Chạy sau Task 1 trong cùng runtime. `gather()` chỉ đọc các file theo input/include
reachable từ root; không nối tất cả file trong thư mục.

- Loại comment `%`, giữ ký tự `\%` và literal text trong verbatim.
- Mở rộng include theo thứ tự, kể cả nhiều tầng; giữ các lần include lặp hợp lệ.
- Chặn vòng lặp bằng active stack, không dùng tập visited để loại lần xuất hiện.
- Cache nội dung đã đọc để giảm truy cập Drive.
- Gộp xong mới tách preamble/body theo begin/end document.
- File thiếu, vòng lặp và đường dẫn macro được giữ nguyên lệnh trong output và báo lỗi.

Chính sách đường dẫn: input/include ưu tiên thư mục import đang hoạt động hoặc working directory
của root; sau đó mới thử thư mục file gọi như fallback và cảnh báo nếu mơ hồ.
import/path và subimport giữ ngữ cảnh thư mục riêng. subfile chỉ đưa body của tài liệu con vào,
không ghép lại preamble và các document marker của nó.


## 7. Định nghĩa gather

Giới hạn: chưa thực thi macro/điều kiện TeX và includeonly, chưa hỗ trợ lệnh include tự định nghĩa. Đây là bộ quét lexical, không phải TeX interpreter;
include nằm trong macro chưa gọi cũng có thể bị nhận diện. Với các cấu trúc này,
cần kiểm tra thủ công thay vì coi output là đường biên dịch đã được xác minh.

`line_in_source` là dòng sau bỏ comment, không phải source map gốc.
Mặc định giới hạn độ sâu 64, số lần mở rộng 10.000 và văn bản 20 triệu ký tự.

In [ ]:
def _real_command(text, start):
    """Reject matches beginning at the second slash of an escaped backslash."""
    previous = start - 1
    count = 0
    while previous >= 0 and text[previous] == "\\":
        count += 1
        previous -= 1
    return count % 2 == 0


def _macro_definition_bodies(masked):
    """Recognize bodies of common newcommand forms, using balanced groups.

    Does not evaluate macros, aliases, catcodes, or primitive def/edef forms.
    Offsets refer to the same-length masked source.
    """
    def group(start, opening, closing):
        if start >= len(masked) or masked[start] != opening:
            return None
        depth = 0
        for pos in range(start, len(masked)):
            if not _real_command(masked, pos):
                continue
            if masked[pos] == opening:
                depth += 1
            elif masked[pos] == closing:
                depth -= 1
                if depth == 0:
                    return pos + 1
        return None

    def skip_space(pos):
        while pos < len(masked) and masked[pos].isspace():
            pos += 1
        return pos

    definitions = []
    pattern = re.compile(r"\\(?:newcommand|renewcommand|providecommand|DeclareRobustCommand)\*?(?![A-Za-z])")
    for match in pattern.finditer(masked):
        if not _real_command(masked, match.start()):
            continue
        pos = skip_space(match.end())
        if pos < len(masked) and masked[pos] == "{":
            end = group(pos, "{", "}")
            if end is None:
                continue
            macro_name = masked[pos + 1:end - 1].strip()
            if not re.fullmatch(r"\\[A-Za-z]+", macro_name):
                continue
            pos = end
        else:
            macro = re.match(r"\\[A-Za-z]+", masked[pos:])
            if not macro:
                continue
            macro_name = macro.group()
            pos += macro.end()
        pos = skip_space(pos)
        for _ in range(2):
            if pos < len(masked) and masked[pos] == "[":
                end = group(pos, "[", "]")
                if end is None:
                    break
                pos = skip_space(end)
        end = group(pos, "{", "}")
        if end is not None:
            definitions.append((pos + 1, end - 1, macro_name))
    return definitions


def gather(source_dir, root_file, *, max_depth=64,
           max_expansions=10000, max_chars=20_000_000):
    """Expand active source includes without writing files or interpreting TeX macros.

    Report statuses: ok, partial (review needed), no_document, or error (required
    source missing, unreadable, cyclic, or truncated by an expansion limit).
    """
    source_dir = Path(source_dir).expanduser().resolve()
    if not source_dir.is_dir():
        raise NotADirectoryError(source_dir)
    if root_file is None:
        raise ValueError("root_file is None; run find_root() successfully first.")
    if min(max_depth, max_expansions, max_chars) <= 0:
        raise ValueError("Gather limits must be positive.")

    root = Path(root_file).expanduser()
    root = (root if root.is_absolute() else source_dir / root).resolve()
    if not root.is_relative_to(source_dir):
        raise ValueError("Root must be inside this version's source directory.")
    if not root.is_file():
        raise FileNotFoundError(root)

    result = {
        "status": "ok", "source_dir": str(source_dir),
        "root_path": root.relative_to(source_dir).as_posix(),
        "expanded_text": "", "preamble": "", "body": "",
        "trailing_text": "", "document_found": False,
        "files_read": [], "include_events": [], "issues": [],
        "warnings": [], "notes": [], "needs_macro_processing": False,
        "expanded_char_count": 0,
    }
    cache, trailing, inspected_controls = {}, [], set()
    expansions = 0
    document_ended = False

    def name(path):
        return path.relative_to(source_dir).as_posix()

    def issue(code, message, severity="warning", **details):
        result["issues"].append({"code": code, "message": message, "severity": severity, **details})
        destination = "notes" if severity == "info" else "warnings"
        result[destination].append(f"[{code}] {message}")

    class GatherLimitError(RuntimeError):
        pass

    def inspect_controls(text, path, start, stop):
        masked = _mask_non_code(text[:stop])
        definitions = _macro_definition_bodies(masked)
        controls = re.compile(r"\\(?:includeonly|iffalse|iftrue|ifdefined|ifx|ifnum|ifdim|ifcase)\b")
        for control in controls.finditer(masked, start):
            pos = control.start()
            if not _real_command(masked, pos) or (path, pos) in inspected_controls:
                continue
            inspected_controls.add((path, pos))
            definition = next((item for item in definitions if item[0] <= pos < item[1]), None)
            line = text.count("\n", 0, pos) + 1
            details = dict(file=name(path), offset=pos, line_in_source=line, command=control.group())
            if definition is not None and control.group() == r"\ifdim":
                body = masked[definition[0]:definition[1]]
                tokens, _ = _source_tokens(body)
                if not tokens and not re.search(r"\\(?:includeonly|csname)\b", body):
                    result["needs_macro_processing"] = True
                    issue("macro_control_deferred",
                          f"{name(path)}:{line}: {control.group()} inside definition {definition[2]}; "
                          "preserved for macro processing (Member 4), not evaluated by gather.",
                          severity="info", **details)
                    continue
            issue("tex_control",
                  f"{name(path)}:{line}: unsupported {control.group()}; compilation/content path needs review.",
                  **details)

    def expand(path, stack, import_dir=None, subfile_mode=False):
        nonlocal expansions, document_ended
        if path not in cache:
            raw, encoding_warning = _read_tex(path)
            cache[path] = raw, *_source_tokens(raw)
            result["files_read"].append(name(path))
            if encoding_warning:
                issue("encoding_fallback", f"{name(path)}: {encoding_warning}", severity="info")

        text, tokens, stop = cache[path]
        start, stop, invalid_subfile = _source_window(tokens, stop, subfile_mode)
        if invalid_subfile:
            issue("invalid_subfile", f"{name(path)} has no complete subfile document body.", severity="error")
            if start == 0:
                return ""

        chunks, length = [], 0

        def append(chunk):
            nonlocal length
            length += len(chunk)
            if length > max_chars:
                raise GatherLimitError(f"Expanded text exceeds max_chars={max_chars}.")
            chunks.append(chunk)

        cursor = start
        for spec in tokens:
            if not start <= spec["start"] < stop:
                continue
            command = spec["command"]
            if command == "end_document":
                append(_strip_tex_comments(text[cursor:spec["end"]]))
                cursor = spec["end"]
                document_ended = True
                break
            if command == "endinput":
                append(_strip_tex_comments(text[cursor:spec["start"]]))
                cursor = spec["end"]
                continue
            if command not in _SOURCE_SIMPLE | _SOURCE_IMPORT:
                continue
            append(_strip_tex_comments(text[cursor:spec["start"]]))
            cursor = spec["end"]
            original = text[spec["start"]:cursor]
            argument = spec["argument"]
            event = {
                "source": name(path), "argument": argument, "command": command,
                "directory": spec["directory"],
                "line_in_source": text.count("\n", 0, spec["start"]) + 1,
                "target": None, "status": "pending",
            }
            result["include_events"].append(event)
            resolved = resolve_source_include(spec, path, source_dir, root.parent, import_dir)
            targets, failure = resolved["targets"], resolved["error"]
            if not targets:
                event["status"] = failure
                severity = "error" if failure in {"missing_file", "outside_source"} and not argument.lower().endswith(".bbl") else "warning"
                issue(failure, f"{name(path)}: cannot resolve {original!r}.", severity=severity)
                append(_strip_tex_comments(original))
                continue
            target = targets[0]
            event["target"] = name(target)
            if len(targets) > 1:
                event["resolution_candidates"] = [name(p) for p in targets]
                issue("include_precedence", f"{name(path)}: chose {name(target)} for {argument!r} by path precedence.",
                      severity="info")
            if target in stack:
                event["status"] = "cycle"
                issue("cycle", " -> ".join(name(p) for p in (*stack, target)), severity="error")
                append(_strip_tex_comments(original))
                continue
            if len(stack) >= max_depth:
                event["status"] = "depth_limit"
                issue("depth_limit", f"Cannot expand {name(target)} at max_depth={max_depth}.", severity="error")
                append(_strip_tex_comments(original))
                continue
            expansions += 1
            if expansions > max_expansions:
                event["status"] = "expansion_limit"
                raise GatherLimitError(f"Exceeded max_expansions={max_expansions}.")
            try:
                content = expand(target, (*stack, target), resolved["import_dir"], command in _SUBFILE_COMMANDS)
            except OSError as exc:
                event["status"] = "read_error"
                issue("read_error", f"{name(target)}: {exc}", severity="error")
                append(_strip_tex_comments(original))
            else:
                event["status"] = "expanded"
                append(content + ("\n" if content and not content.endswith("\n") and not document_ended else ""))
            if document_ended:
                break

        if document_ended:
            if cursor < stop:
                trailing.append(text[cursor:stop])
        else:
            append(_strip_tex_comments(text[cursor:stop]))
        inspect_controls(text, path, start, cursor if document_ended else stop)
        return "".join(chunks)

    try:
        expanded = expand(root, (root,))
    except (OSError, GatherLimitError) as exc:
        issue("gather_error", str(exc), severity="error")
        for event in result["include_events"]:
            if event["status"] == "pending":
                event["status"] = "aborted"
        result["status"] = "error"
        return result

    result["expanded_text"] = expanded
    result["expanded_char_count"] = len(expanded)
    result["trailing_text"] = "".join(trailing)
    tokens, _ = _source_tokens(expanded)
    begins = [token for token in tokens if token["command"] == "begin_document"]
    if not begins:
        result["preamble"] = expanded
        result["status"] = "error" if any(item["severity"] == "error" for item in result["issues"]) else "no_document"
        issue("missing_begin_document", "No begin{document} after gathering.")
        return result

    begin = begins[0]
    result["document_found"] = True
    result["preamble"] = expanded[:begin["start"]]
    if len(begins) > 1:
        issue("multiple_documents", "Multiple begin{document} markers; inspect compilation path.")
    end = next((token for token in tokens if token["command"] == "end_document"
                and token["start"] >= begin["end"]), None)
    result["body"] = expanded[begin["end"]:end["start"] if end else len(expanded)]
    if end is None:
        issue("missing_end_document", "No end{document}; retained body through EOF.")
    severities = {item["severity"] for item in result["issues"]}
    result["status"] = "error" if "error" in severities else "partial" if "warning" in severities else "ok"
    return result


## 8. Chạy gather và đọc báo cáo

- `ok`: không phát hiện vấn đề ở các cấu trúc được hỗ trợ (không đảm bảo tương đương PDF).
  `ifdim` trong định nghĩa macro nhận diện được, không chứa lệnh file/document trực tiếp,
  được giữ nguyên và ghi vào `notes` cho Member 4; không làm gather thành partial.
- `partial`: có output nhưng còn include/control flow chưa xử lý chắc chắn; kiểm tra warnings.
- `no_document`: không thấy begin document sau gom; chưa có body sử dụng được.
- `error`: lỗi đọc root hoặc vượt giới hạn; không sử dụng output để dựng cây.

Đây là trạng thái nội bộ của gather, không tự ghi đè `parse_status` cuối cùng của Member 1.

In [ ]:
gather_result = None

if root_file is None:
    print("Không có root hợp lệ. Kiểm tra Task 1 trước.")
else:
    gather_result = gather(source_dir, root_file)
    print("Gather status:", gather_result["status"])
    print("Root:", gather_result["root_path"])
    print("Document found:", gather_result["document_found"])
    print("Unique files read:", len(gather_result["files_read"]))
    print("Include occurrences:", len(gather_result["include_events"]))
    print("Expanded characters:", gather_result["expanded_char_count"])
    print("Preamble characters:", len(gather_result["preamble"]))
    print("Body characters:", len(gather_result["body"]))
    print("Needs macro processing:", gather_result["needs_macro_processing"])
    print("\nFiles read:")
    for file in gather_result["files_read"]:
        print("-", file)
    print("\nWarnings:")
    if not gather_result["warnings"]:
        print("(none)")
    for warning in gather_result["warnings"]:
        print("-", warning)

    print("\nNotes for macro processing (Member 4):")
    if not gather_result["notes"]:
        print("(none)")
    for note in gather_result["notes"]:
        print("-", note)

    events = gather_result["include_events"]
    if events:
        try:
            import pandas as pd
            from IPython.display import display
        except ImportError:
            for event in events[:30]:
                print(event)
        else:
            display(pd.DataFrame(events).head(30))

## 9. Xem nội dung và chuyển cho bước sau

Preamble giữ định nghĩa macro cho Member 4. Body giữ LaTeX và citation để chuẩn hóa
và dựng cây sau này; chưa xóa bibliography tại Task 2 vì Member 3 còn cần trích xuất.
`expanded_text` giữ toàn bộ source sau gom; `trailing_text` là phần sau end document.
Không có file nào được ghi lên Drive khi chạy các cell này.

In [ ]:
preamble = None
document_body = None

if gather_result is not None and gather_result["document_found"]:
    print("--- PREAMBLE: tối đa 1500 ký tự ---")
    print(gather_result["preamble"][:1500])
    print("\n--- BODY: tối đa 4000 ký tự ---")
    print(gather_result["body"][:4000])

    if gather_result["status"] == "ok":
        preamble = gather_result["preamble"]
        document_body = gather_result["body"]
        print("\nĐã đặt preamble và document_body cho bước tiếp theo.")
    else:
        print("\nOutput chưa đầy đủ: kiểm tra warnings trước khi chuyển sang dựng cây.")
else:
    print("Chưa có document body. Xem kết quả Task 1 và báo cáo gather.")

## Kiểm tra với dữ liệu nhóm

1. Chạy paper `2609-37833/v1`: nếu chỉ có main.tex và không có input/include, số lần include bằng 0 là hợp lệ.
2. Thử version có nhiều file được input/include: xem target và thứ tự các event.
3. Kiểm tra body không chứa preamble, còn các section, công thức, figure và bibliography cần xử lý tiếp.
4. Chọn một file .tex không được include trong version nếu có; xác nhận file đó không nằm trong files_read.
5. Gửi Gather status, số file/lần include và warnings để kiểm tra các trường hợp thật.

Khi đổi paper/version, chạy lại từ cell cấu hình, Task 1 rồi Task 2 để tránh dùng biến cũ.
Trước khi commit, xóa output dài/đường dẫn cá nhân; không commit dữ liệu M1.

# Task 3 — Dựng cây nội dung

Chạy sau Task 2 trong cùng runtime. Không cần cài thêm thư viện cho parser.
Các node dùng ID tạm theo lần xuất hiện; không dedup, không cấp element_id cuối cùng.

Hỗ trợ heading/star/short title, Abstract, Appendix, Acknowledgements, References,
công thức khối, figure/table, itemize lồng nhau (kể cả enumerate/description).
Môi trường chưa biết và nội dung lỗi được giữ lại kèm cảnh báo để kiểm tra.

Khi chưa có hàm Member 4, vẫn dựng được cây trung gian với text_block.
Khi đó kiểm thử cấu trúc vẫn chạy được, nhưng chưa công nhận các lá văn xuôi là câu.

## 10. Hàm dựng cây và kiểm tra cấu trúc

Mỗi node gồm node_id, parent_id, kind, content, order, source_span, metadata.
Heading chứa tiêu đề; container khác có content rỗng; nội dung nằm ở các node con.
Formula/figure giữ source thô. source_span trỏ vào source_text sau expand macro và bỏ comment;
span của sentence trỏ về text block gốc, không phải vị trí từng câu sau chuẩn hóa.

Môi trường có `\end` lệch cặp bên trong: thử khớp `\end` cùng tên; không có `\end` nào thì bỏ marker
`\begin` và parse tiếp (có cảnh báo), không bỏ cả version. `\bibitem` viết ngoài `thebibliography`
bị loại như danh sách tài liệu tham khảo.

Đây là parser lexical, không thay thế TeX. Không diễn giải catcode, macro tự định nghĩa,
conditional hoặc cấu trúc phát sinh từ macro nếu chưa có hook expand.
Validator kiểm tra cấu trúc ID/parent/order/cycle, không chứng minh ngữ nghĩa hay độ phủ toàn corpus.

In [ ]:
from collections import Counter
from functools import lru_cache
import re


_TREE_TOKEN_START_RE = re.compile(r"[{$\\]")
_TREE_MATH_STOP_RE = re.compile(r"[\\$]")
_TREE_GROUP_STOPS = {}   # opening + closing -> pattern of the characters a group scan must look at
_TREE_BRACE_TOKEN_RE = re.compile(r"\\(?:[A-Za-z]+\*?|[^\r\n])|[{}]")
_TREE_UNCLOSED = {}      # text with an unclosed {group} -> _brace_table of it
_TREE_LOOSE_BIBITEM_END = re.compile(
    r"\\bibitem(?![A-Za-z])|\n[ \t]*\n|\\end\s*\{"
    r"|\\(?:part|chapter|section|subsection|subsubsection|paragraph|appendix)(?![A-Za-z])")
_TREE_COMMAND = re.compile(r"\\([A-Za-z]+\*?|[^\r\n])")
_TREE_ENV = re.compile(r"\\(begin|end)\s*\{([^{}]+)\}")
_TREE_HEADINGS = {
    "part": 1, "chapter": 1, "section": 2, "subsection": 3,
    "subsubsection": 4, "paragraph": 5, "subparagraph": 6,
}
_TREE_FORMULAS = {
    "equation", "equation*", "align", "align*", "alignat", "alignat*",
    "gather", "gather*", "multline", "multline*", "displaymath", "eqnarray", "eqnarray*", "flalign", "flalign*", "dmath", "dmath*",
}
_TREE_FIGURES = {
    "figure", "table", "longtable", "wrapfigure", "wraptable", "sidewaysfigure", "sidewaystable",
    "algorithm", "algorithm2e", "algorithmic", "tabular", "tabularx", "tabulary",
    "tikzpicture", "SCfigure", "deluxetable", "planotable", "listing", "subtable", "subfigure",
}
_TREE_LISTS = {
    "itemize", "enumerate", "description", "compactitem", "compactenum", "compactdesc",
    "inparaenum", "asparaenum", "asparaitem", "inparaitem",
}
_TREE_LITERAL = {"verbatim", "verbatim*", "Verbatim", "lstlisting", "minted"}
_TREE_CONTAINERS = {
    "center", "flushleft", "flushright", "quote", "quotation", "proof",
    "theorem", "lemma", "proposition", "corollary", "definition", "remark",
    "example", "acknowledgments", "acknowledgements", "acknowledgement",
}


def _tree_group(text, start, opening="{", closing="}"):
    """Return exclusive closing position of a balanced group, or None."""
    if start >= len(text) or text[start] != opening:
        return None
    if opening == "{" and text in _TREE_UNCLOSED:   # see the end of this function
        return _TREE_UNCLOSED[text].get(start)
    depth = 0
    braces = 0
    pos = start
    stops = _TREE_GROUP_STOPS.get(opening + closing)
    if stops is None:
        stops = _TREE_GROUP_STOPS[opening + closing] = re.compile(
            "[" + re.escape("\\{}" + opening + closing) + "]")
    while pos < len(text):
        marker = stops.search(text, pos)   # other characters change nothing
        if marker is None:
            break
        pos = marker.start()
        if text[pos] == "\\":
            match = _TREE_COMMAND.match(text, pos)
            if match:
                pos = match.end()
                continue
        char = text[pos]
        if opening == "[":
            if char == "{":
                braces += 1
            elif char == "}":
                braces -= 1
        if not braces:
            if char == opening:
                depth += 1
            elif char == closing:
                depth -= 1
                if depth == 0:
                    return pos + 1
        pos += 1
    if opening == "{":   # unclosed: match every brace of this text once (see _brace_table)
        if len(_TREE_UNCLOSED) > 4:
            _TREE_UNCLOSED.clear()
        _TREE_UNCLOSED[text] = _brace_table(text, _TREE_BRACE_TOKEN_RE)
    return None


def _tree_skip_space(text, pos):
    while pos < len(text) and text[pos].isspace():
        pos += 1
    return pos


def _tree_heading(text, command):
    pos = _tree_skip_space(text, command.end())
    short_title = None
    if pos < len(text) and text[pos] == "[":
        end = _tree_group(text, pos, "[", "]")
        if end is None:
            return None
        short_title = text[pos + 1:end - 1]
        pos = _tree_skip_space(text, end)
    end = _tree_group(text, pos)
    if end is None:
        return None
    return text[pos + 1:end - 1], short_title, end


def _tree_protected_end(text, pos):
    """Skip inline math / verb so their contents cannot create headings/items."""
    if text.startswith(r"\verb", pos):
        match = _SOURCE_VERB_RE.match(text, pos)   # no copy of the rest of the text
        if match:
            start = match.end()
            if start < len(text) and not text[start].isspace():
                end = text.find(text[start], start + 1)
                line = text.find("\n", start + 1)
                if end >= 0 and (line < 0 or end < line):
                    return end + 1
    if text.startswith(r"\(", pos):
        end = text.find(r"\)", pos + 2)
        return len(text) if end < 0 else end + 2
    if text[pos:pos + 1] == "$" and not text.startswith("$$", pos):
        end = pos + 1
        while True:
            marker = _TREE_MATH_STOP_RE.search(text, end)   # next backslash or $
            if marker is None:
                return len(text)
            end = marker.start()
            if text[end] == "\\":
                match = _TREE_COMMAND.match(text, end)
                end = match.end() if match else end + 1
            else:
                return end + 1
    return None


def _tree_environment_end(text, begin):
    """Stack match first. If an inner environment is malformed (an \\end that does not close
    the innermost \\begin), the matching \\end of the same name, counting only that name's
    nesting, so one bad inner block does not lose the whole version."""
    strict = _tree_environment_end_strict(text, begin)
    if strict is not None:
        return strict
    pattern = re.compile(r"\\(begin|end)\s*\{" + re.escape(begin.group(2)) + r"\}")
    depth = 1
    for match in pattern.finditer(text, begin.end()):
        depth += 1 if match.group(1) == "begin" else -1
        if depth == 0:
            return match.start(), match.end()
    return None


def _tree_environment_end_strict(text, begin):
    """Match environments using a stack; literal environments are opaque."""
    env_name = begin.group(2)
    if env_name in _TREE_LITERAL or env_name == "comment":
        end = re.compile(r"\\end\s*\{" + re.escape(env_name) + r"\}").search(text, begin.end())
        return None if end is None else (end.start(), end.end())
    stack = [env_name]
    pos = begin.end()
    while pos < len(text):
        marker = _TREE_TOKEN_START_RE.search(text, pos)   # only $, { and backslash matter
        if marker is None:
            break
        pos = marker.start()
        if text.startswith("$$", pos) or text.startswith(r"\[", pos):
            delimiter = "$$" if text.startswith("$$", pos) else r"\]"
            end = text.find(delimiter, pos + 2)
            pos = len(text) if end < 0 else end + 2
            continue
        protected = _tree_protected_end(text, pos)
        if protected is not None:
            pos = protected
            continue
        if text[pos] == "{":
            group_end = _tree_group(text, pos)
            if group_end:
                pos = group_end
                continue
        if text[pos] == "\\":
            env = _TREE_ENV.match(text, pos)
            if env:
                if env.group(1) == "begin":
                    if env.group(2) in _TREE_LITERAL | {"comment"}:
                        end = _tree_environment_end(text, env)
                        if end is None:
                            return None
                        pos = end[1]
                        continue
                    stack.append(env.group(2))
                else:
                    if env.group(2) != stack[-1]:
                        return None
                    stack.pop()
                    if not stack:
                        return env.start(), env.end()
                pos = env.end()
                continue
            command = _TREE_COMMAND.match(text, pos)
            if command:
                pos = command.end()
                continue
        pos += 1
    return None


def validate_tree(nodes):
    """Validate occurrence IDs, parents, connectivity, order and leaf types."""
    errors = []
    ids = [node["node_id"] for node in nodes]
    index = {node["node_id"]: node for node in nodes}
    if len(index) != len(nodes):
        errors.append("Duplicate occurrence node_id")
    roots = [node for node in nodes if node["parent_id"] is None]
    if len(roots) != 1 or roots[0]["kind"] != "document":
        errors.append("Expected exactly one document root")
    children = {}
    for node in nodes:
        parent = node["parent_id"]
        if parent is not None:
            if parent not in index:
                errors.append(f"Missing parent: {node['node_id']}")
            children.setdefault(parent, []).append(node)
        seen = set()
        cursor = node
        while cursor["parent_id"] is not None:
            if cursor["node_id"] in seen:
                errors.append(f"Cycle: {node['node_id']}")
                break
            seen.add(cursor["node_id"])
            if cursor["parent_id"] not in index:
                break
            cursor = index[cursor["parent_id"]]
    for parent, siblings in children.items():
        orders = sorted(node["order"] for node in siblings)
        if orders != list(range(len(siblings))):
            errors.append(f"Invalid sibling order under {parent}")
    leaf_kinds = {"sentence", "formula", "figure", "text_block"}
    for node in nodes:
        if node["kind"] in leaf_kinds and node["node_id"] in children:
            errors.append(f"Leaf has children: {node['node_id']}")
    return {"valid": not errors, "errors": sorted(set(errors))}


def build_tree_report(document_body, *, preamble="", macro_expander=None,
               text_normalizer=None, sentence_splitter=None,
               macro_processing_required=False, max_depth=64):
    """
    Parse one version into occurrence nodes. Does not deduplicate or assign element_id.

    Optional Member 4 contracts:
      macro_expander(preamble: str, body: str) -> str (BODY only)
      text_normalizer(raw_prose: str) -> str
      sentence_splitter(normalized_prose: str) -> list[str]

    No splitter => intermediate text_block nodes (never silently called sentences).
    Formula/figure content stays raw for subsequent normalization.
    Offsets refer to source_text AFTER macro expansion and comment removal.
    """
    if not isinstance(document_body, str):
        raise TypeError("document_body must be str")
    if max_depth < 1:
        raise ValueError("max_depth must be positive")
    source = document_body
    if macro_expander is not None:
        source = macro_expander(preamble, source)
        if not isinstance(source, str):
            raise TypeError("macro_expander must return a body string")
    source = _strip_tex_comments(source)
    nodes, issues, excluded = [], [], []
    sibling_counts = Counter()
    appendix = False

    def issue(code, message, start, severity="warning"):
        issues.append({"code": code, "message": message, "start": start,
                       "severity": severity})

    def add(kind, content, parent, start, end, **metadata):
        node_id = f"n{len(nodes)}"
        order = sibling_counts[parent]
        sibling_counts[parent] += 1
        nodes.append({"node_id": node_id, "parent_id": parent, "kind": kind,
                      "content": content, "order": order,
                      "source_span": [start, end], "metadata": metadata})
        return node_id

    root = add("document", "", None, 0, len(source))

    def prose(raw, parent, start):
        if not raw.strip():
            return
        # A blank line is a prose-block boundary, not a final hierarchy level.
        for part in re.finditer(r"\S[\s\S]*?(?=\n[ \t]*\n|\Z)", raw):
            content = part.group().strip()
            if not content:
                continue
            begin = start + part.start()
            finish = start + part.end()
            if text_normalizer is not None:
                content = text_normalizer(content)
                if not isinstance(content, str):
                    raise TypeError("text_normalizer must return str")
            if not content.strip():
                excluded.append({"kind": "normalized_empty", "source_span": [begin, finish]})
                continue
            if sentence_splitter is None:
                add("text_block", content, parent, begin, finish)
            else:
                sentences = sentence_splitter(content)
                if (not isinstance(sentences, list) or not sentences
                    or any(not isinstance(s, str) or not s.strip() for s in sentences)):
                    raise ValueError("sentence_splitter must return a nonempty list of nonempty strings")
                for sentence in sentences:
                    # Precise spans cannot be recovered after arbitrary normalization.
                    add("sentence", sentence.strip(), parent, begin, finish,
                        span_scope="source_text_block")

    def list_body(text, parent, offset, depth):
        items = []
        pos = 0
        while pos < len(text):
            marker = _TREE_TOKEN_START_RE.search(text, pos)   # only $, { and backslash matter
            if marker is None:
                break
            pos = marker.start()
            protected = _tree_protected_end(text, pos)
            if protected is not None:
                pos = protected
                continue
            if text.startswith("$$", pos) or text.startswith(r"\[", pos):
                delimiter = "$$" if text.startswith("$$", pos) else r"\]"
                end = text.find(delimiter, pos + 2)
                pos = len(text) if end < 0 else end + 2
                continue
            if text[pos] == "{":
                end = _tree_group(text, pos)
                if end:
                    pos = end
                    continue
            if text[pos] == "\\":
                env = _TREE_ENV.match(text, pos)
                if env and env.group(1) == "begin":
                    end = _tree_environment_end(text, env)
                    if end:
                        pos = end[1]
                        continue
                command = _TREE_COMMAND.match(text, pos)
                if command:
                    if command.group(1) == "item":
                        start_content = _tree_skip_space(text, command.end())
                        label = None
                        if start_content < len(text) and text[start_content] == "[":
                            end = _tree_group(text, start_content, "[", "]")
                            if end:
                                label = text[start_content + 1:end - 1]
                                start_content = end
                        items.append((pos, start_content, label))
                        pos = start_content
                    else:
                        pos = command.end()
                    continue
            pos += 1
        prefix_end = items[0][0] if items else len(text)
        if text[:prefix_end].strip():
            issue("list_prefix", "Text before first item retained in an implicit item", offset)
            item = add("item", "", parent, offset, offset + prefix_end, implicit=True)
            parse(text[:prefix_end], item, offset, depth + 1, False)
        for index, (start, content_start, label) in enumerate(items):
            end = items[index + 1][0] if index + 1 < len(items) else len(text)
            item = add("item", "", parent, offset + start, offset + end, label=label)
            parse(text[content_start:end], item, offset + content_start, depth + 1, False)

    def parse(text, base_parent, offset, depth, allow_headings=True):
        nonlocal appendix
        if depth > max_depth:
            issue("depth_limit", "Unparsed content retained at recursion limit", offset)
            add("text_block", text, base_parent, offset, offset + len(text), unparsed=True)
            return
        stack = [(0, base_parent)]
        reference_rank = None
        reference_start = None
        cursor = pos = 0

        def flush(end):
            if reference_rank is None:
                prose(text[cursor:end], stack[-1][1], offset + cursor)

        def close_references(end):
            nonlocal reference_rank, reference_start
            if reference_rank is not None:
                excluded.append({"kind": "references_section",
                                 "source_span": [offset + reference_start, offset + end]})
                reference_rank = reference_start = None

        while pos < len(text):
            marker = _TREE_TOKEN_START_RE.search(text, pos)
            if marker is None:
                break
            pos = marker.start()
            protected = _tree_protected_end(text, pos)
            if protected is not None:
                pos = protected
                continue
            if text.startswith("$$", pos) or text.startswith(r"\[", pos):
                flush(pos)
                delimiter = "$$" if text.startswith("$$", pos) else r"\]"
                end = text.find(delimiter, pos + 2)
                if end < 0:
                    end = len(text)
                    issue("unclosed_formula", "Unclosed display math retained", offset + pos)
                else:
                    end += 2
                if reference_rank is None:
                    add("formula", text[pos:end], stack[-1][1], offset + pos, offset + end,
                        environment="dollars" if delimiter == "$$" else "brackets")
                cursor = pos = end
                continue
            if text[pos] == "{":
                end = _tree_group(text, pos)
                if end:
                    pos = end
                    continue
                issue("unclosed_group", "Unbalanced brace; content retained", offset + pos)
            if text[pos] != "\\":
                pos += 1
                continue
            command = _TREE_COMMAND.match(text, pos)
            if not command:
                pos += 1
                continue
            name = command.group(1)
            plain_name = name.rstrip("*")
            if plain_name in _TREE_HEADINGS and allow_headings:
                heading = _tree_heading(text, command)
                if heading is None:
                    issue("malformed_heading", f"Cannot read {name} title", offset + pos)
                    pos = command.end()
                    continue
                title, short, end = heading
                rank = _TREE_HEADINGS[plain_name]
                flush(pos)
                if reference_rank is not None and rank > reference_rank:
                    cursor = pos = end
                    continue
                close_references(pos)
                plain_title = re.sub(r"\\(?:textbf|textit|textrm|textnormal|emph)\b", "", title)
                normalized_title = re.sub(r"[^a-z]", "", plain_title.casefold())
                if title.strip() in {r"\refname", r"\bibname"} or normalized_title in {
                    "references", "bibliography", "workscited", "literaturecited",
                }:
                    reference_rank, reference_start = rank, pos
                    cursor = pos = end
                    continue
                while len(stack) > 1 and stack[-1][0] >= rank:
                    stack.pop()
                node = add("heading", title, stack[-1][1], offset + pos, offset + end,
                           command=plain_name, level=rank, starred=name.endswith("*"),
                           short_title=short, is_appendix=appendix)
                stack.append((rank, node))
                cursor = pos = end
                continue
            if plain_name in {"footnote", "footnotetext"}:
                note = _tree_heading(text, command)
                if note is not None:
                    content, _, end = note
                    flush(pos)
                    if reference_rank is None:
                        parse(content, stack[-1][1], offset + end - len(content) - 1,
                              depth + 1, allow_headings=False)
                    cursor = pos = end
                    continue
            if name == "bibitem":
                # \bibitem written without thebibliography: a reference, not text. Excluded
                # up to the next \bibitem, blank line, heading or \end.
                flush(pos)
                stop_at = _TREE_LOOSE_BIBITEM_END.search(text, command.end())
                end = stop_at.start() if stop_at else len(text)
                excluded.append({"kind": "bibitem", "source_span": [offset + pos, offset + end]})
                cursor = pos = end
                continue
            if name == "appendix":
                flush(pos)
                close_references(pos)
                appendix = True
                stack = [(0, base_parent)]
                cursor = pos = command.end()
                continue
            env = _TREE_ENV.match(text, pos)
            if env and env.group(1) == "begin":
                flush(pos)
                end_info = _tree_environment_end(text, env)
                if end_info is None:
                    # No \end at all: drop the \begin marker and parse on, instead of
                    # turning the rest of the version into one unparsed block.
                    issue("unclosed_environment", f"Cannot match environment {env.group(2)}", offset + pos)
                    cursor = pos = env.end()
                    continue
                closing_start, end = end_info
                env_name = env.group(2).strip()
                base_env = env_name.rstrip("*")
                parent = stack[-1][1]
                if reference_rank is not None or base_env in {"thebibliography", "comment", "titlepage", "filecontents"}:
                    if reference_rank is None:
                        excluded.append({"kind": env_name, "source_span": [offset + pos, offset + end]})
                elif base_env in _TREE_FORMULAS | _TREE_FIGURES:
                    kind = "formula" if base_env in _TREE_FORMULAS else "figure"
                    add(kind, text[pos:end], parent, offset + pos, offset + end, environment=env_name)
                elif env_name in _TREE_LITERAL:
                    issue("literal_environment", f"Literal {env_name} retained for review", offset + pos)
                    add("text_block", text[pos:end], parent, offset + pos, offset + end, protected=True, environment=env_name)
                else:
                    kind = "abstract" if env_name == "abstract" else "itemize" if base_env in _TREE_LISTS else "environment"
                    container = add(kind, "", parent, offset + pos, offset + end, environment=env_name)
                    inner = text[env.end():closing_start]
                    if base_env in _TREE_LISTS:
                        list_body(inner, container, offset + env.end(), depth)
                    else:
                        if env_name not in _TREE_CONTAINERS | {"abstract", "appendices"}:
                            issue("unknown_environment", f"Preserved container {env_name}; inspect hierarchy", offset + pos)
                        previous_appendix = appendix
                        if env_name == "appendices":
                            appendix = True
                        parse(inner, container, offset + env.end(), depth + 1,
                              allow_headings=env_name != "abstract")
                        if env_name == "appendices":
                            appendix = previous_appendix
                cursor = pos = end
                continue
            if name in {"bibliography", "printbibliography", "bibliographystyle"}:
                flush(pos)
                end = _tree_skip_space(text, command.end())
                if end < len(text) and text[end] == "[":
                    end = _tree_group(text, end, "[", "]") or end
                    end = _tree_skip_space(text, end)
                if end < len(text) and text[end] == "{":
                    end = _tree_group(text, end) or end
                excluded.append({"kind": name, "source_span": [offset + pos, offset + end]})
                cursor = pos = end
                continue
            if name in {"input", "include", "includeonly", "subfile", "subfileinclude",
                        "import", "subimport", "inputfrom", "subinputfrom", "includefrom", "subincludefrom"}:
                issue("unresolved_include", f"{name} reached build_tree_report; inspect gather", offset + pos)
            if env and env.group(1) == "end":
                issue("unexpected_end", f"Unexpected end of {env.group(2)}", offset + pos)
            pos = command.end()
        flush(len(text))
        close_references(len(text))

    parse(source, root, 0, 0)
    parents = {node["parent_id"] for node in nodes}
    for node in nodes:
        if node["kind"] in {"document", "heading", "abstract", "itemize", "item", "environment"} and node["node_id"] not in parents:
            issue("empty_container", f"Empty {node['kind']} node {node['node_id']}; inspect source", node["source_span"][0])
    validation = validate_tree(nodes)
    counts = dict(Counter(node["kind"] for node in nodes))
    if sentence_splitter is None:
        issue("sentence_hook_missing", "Intermediate text_block leaves; connect Member 4 sentence splitter", 0, "info")
    if macro_processing_required and macro_expander is None:
        issue("macro_hook_missing", "Gather identified deferred macros; connect Member 4 macro expander", 0, "info")
    warning_free = not any(item["severity"] == "warning" for item in issues)
    leaves_complete = counts.get("text_block", 0) == 0
    integrated = (sentence_splitter is not None and text_normalizer is not None
                  and (macro_expander is not None or not macro_processing_required))
    return {
        "nodes": nodes, "issues": issues, "excluded": excluded,
        "source_text": source, "counts": counts, "validation": validation,
        "structure_valid": validation["valid"],
        "sentence_leaves_complete": leaves_complete,
        "member4_hooks_integrated": integrated,
        "automatic_checks_passed": validation["valid"] and warning_free,
        "ready_for_manual_review": validation["valid"] and warning_free,
        "pipeline_candidate": validation["valid"] and warning_free and leaves_complete and integrated,
    }


_PIPELINE_FRONT_RE = re.compile(r"\\(?:title|icmltitle|abstract)\*?(?![A-Za-z@])")
_PIPELINE_ABSTRACT_ENV_RE = re.compile(r"\\begin\s*\{abstract\}")
_PIPELINE_LABEL_RE = re.compile(r"\\(?:label|thanks|footnote|footnotemark)\*?(?![A-Za-z@])")
_PIPELINE_TEXT_RE = re.compile(r"\\[A-Za-z@]+\*?")


@lru_cache(maxsize=4)
def _pipeline_splitter(notebook_path):
    member11, _ = load_member11_functions(notebook_path)
    return member11.split_sentences


def _normalization_notebook():
    for path in (Path("11_normalize.ipynb"), Path("notebooks/11_normalize.ipynb"), Path("src/11_normalize.ipynb")):
        if path.is_file():
            return path.resolve()
    raise FileNotFoundError("11_normalize.ipynb is required for the Member 4 sentence splitter")


def _pipeline_front_matter(text):
    """Read top-level title and abstract and convert the abstract command to a block."""
    masked = _mask_non_code(text)
    chunks = []
    pos = cursor = 0
    title = ""
    abstracts = []
    while pos < len(masked):
        marker = _SOURCE_COMMAND_OR_GROUP_RE.search(masked, pos)
        if marker is None:
            break
        pos = marker.start()
        if masked[pos] == "{":
            group = _source_group(masked, pos)
            if group:
                pos = group[1]
                continue
        if masked[pos] != "\\":
            pos += 1
            continue
        match = _PIPELINE_FRONT_RE.match(masked, pos)
        heading = _tree_heading(masked, match) if match else None
        if heading:
            _, _, end = heading
            # Read offsets from the masked view and preserve the original argument.
            original = _tree_heading(text, match)
            value = original[0] if original else heading[0]
            chunks.append(text[cursor:pos])
            if match.group().rstrip("*") == r"\abstract":
                abstract = "\\begin{abstract}" + value + "\\end{abstract}"
                abstracts.append(abstract)
                chunks.append(abstract)
            elif not title:
                title = value
            cursor = pos = end
            continue

        abstract = _PIPELINE_ABSTRACT_ENV_RE.match(masked, pos)
        env = _TREE_ENV.match(masked, pos) if abstract else None
        bounds = _tree_environment_end(masked, env) if env else None
        if bounds:
            abstracts.append(text[pos:bounds[1]])
            pos = bounds[1]
            continue
        token = _SOURCE_COMMAND.match(masked, pos)
        pos = token.end() if token else pos + 1
    chunks.append(text[cursor:])
    return title, abstracts, "".join(chunks)


def _pipeline_title(text):
    """Remove annotation commands from titles without discarding the title text."""
    chunks = []
    cursor = 0
    for match in _PIPELINE_LABEL_RE.finditer(text):
        if match.start() < cursor or not _real_command(text, match.start()):
            continue
        note = _tree_heading(text, match)
        if note:
            end = note[2]
        elif match.group().rstrip("*") == r"\footnotemark":
            end = match.end()
        else:
            continue
        chunks.append(text[cursor:match.start()])
        cursor = end
    chunks.append(text[cursor:])
    return "".join(chunks).replace(r"\&", "&").strip()


def build_tree(text, *, sentence_splitter=None):
    """Return the nested tree expected by the pipeline, from already normalized LaTeX."""
    if not isinstance(text, str):
        raise TypeError("build_tree expects a string")
    tokens, _ = _source_tokens(text)
    begin = next((token for token in tokens if token["command"] == "begin_document"), None)
    if begin:
        end = next((token for token in tokens if token["command"] == "end_document"
                    and token["start"] >= begin["end"]), None)
        preamble = text[:begin["start"]]
        body = text[begin["end"]:end["start"] if end else len(text)]
    else:
        preamble, body = "", text
    title, preamble_abstracts, _ = _pipeline_front_matter(preamble)
    body_title, body_abstracts, body = _pipeline_front_matter(body)
    if preamble_abstracts and not body_abstracts:
        body = "\n".join([*preamble_abstracts, body])
    if sentence_splitter is None:
        sentence_splitter = _pipeline_splitter(_normalization_notebook())
    report = build_tree_report(body, sentence_splitter=sentence_splitter)
    if not report["structure_valid"]:
        raise ValueError(f"Invalid LaTeX tree: {report['validation']['errors']}")

    root = {"kind": "document", "content": _pipeline_title(title or body_title), "children": []}
    exported = {}
    for node in report["nodes"]:
        node_id, parent_id = node["node_id"], node["parent_id"]
        if parent_id is None:
            exported[node_id] = root
            continue
        parent = exported[parent_id]
        kind, content, metadata = node["kind"], node["content"], node["metadata"]
        if kind == "environment":
            exported[node_id] = parent
            continue
        if kind == "heading":
            kind = {"part": "chapter", "subparagraph": "paragraph"}.get(metadata["command"], metadata["command"])
            content = _pipeline_title(content)
        elif kind == "item":
            content = metadata.get("label") or content
        elif kind == "abstract":
            content = "Abstract"
        elif kind == "text_block":
            if not metadata.get("protected"):
                raise ValueError(f"Unparsed LaTeX at offset {node['source_span'][0]}")
            kind = "figure"
        elif kind == "sentence" and not re.search(r"\w", _PIPELINE_TEXT_RE.sub("", content)):
            continue
        child = {"kind": kind, "content": content, "children": []}
        parent["children"].append(child)
        exported[node_id] = child
    return root


def preview_tree(tree_result, max_nodes=80, max_label=90):
    """Return a bounded text preview; parent IDs and orders remain in nodes."""
    nodes = tree_result["nodes"]
    index = {node["node_id"]: node for node in nodes}
    lines = []
    for node in nodes[:max_nodes]:
        depth, parent, seen = 0, node["parent_id"], set()
        while parent in index and parent not in seen:
            seen.add(parent)
            depth += 1
            parent = index[parent]["parent_id"]
        label = re.sub(r"\s+", " ", node["content"]).strip()
        env = node["metadata"].get("environment", "")
        lines.append(f"{'  ' * min(depth, 12)}{node['node_id']} [{node['kind']}] {label[:max_label] or env}")
    if len(nodes) > max_nodes:
        lines.append(f"... {len(nodes) - max_nodes} more nodes")
    return "\n".join(lines)


## 11. Chạy bộ test Task 3

Kỳ vọng **37 passed, 0 failed**. Cell này dùng source giả lập trong bộ nhớ, không sửa Drive.
Các adapter tối giản trong test chỉ kiểm tra giao diện hook, KHÔNG dùng để tách câu paper.
Test thất bại sẽ ném AssertionError; dừng lại và gửi lỗi trước khi tiếp tục.

In [ ]:
def run_build_tree_tests():
    """Synthetic regression tests. No Drive access or Member 4 implementation needed."""
    passed = []

    def run(name, source, check, **kwargs):
        result = build_tree_report(source, **kwargs)
        assert result["structure_valid"], (name, result["validation"])
        assert check(result), (name, result["counts"], result["issues"])
        passed.append(name)
        return result

    def kinds(result, kind):
        return [node for node in result["nodes"] if node["kind"] == kind]

    def content(result):
        return " ".join(node["content"] for node in result["nodes"])

    def warnings(result):
        return [issue for issue in result["issues"] if issue["severity"] == "warning"]

    r = run("heading_stack", r"\section{A}One\subsection{B}Two\section{C}Three",
            lambda r: len(kinds(r, "heading")) == 3)
    headings = kinds(r, "heading")
    assert headings[0]["parent_id"] == headings[2]["parent_id"] == "n0"
    assert headings[1]["parent_id"] == headings[0]["node_id"]
    assert kinds(r, "text_block")[2]["parent_id"] == headings[2]["node_id"]
    run("chapter_and_skipped_levels", r"\chapter{C}\subsection{S}Text",
        lambda r: kinds(r, "heading")[1]["parent_id"] == kinds(r, "heading")[0]["node_id"])
    run("no_section", "Opening text.", lambda r: kinds(r, "text_block")[0]["parent_id"] == "n0")
    run("prefix_text", r"Preface.\section{A}Body", lambda r: kinds(r, "text_block")[0]["parent_id"] == "n0")
    run("star_short_nested_title", r"\section*[Short]{A \textbf{nested} title}Body",
        lambda r: kinds(r, "heading")[0]["metadata"]["starred"] and kinds(r, "heading")[0]["metadata"]["short_title"] == "Short" and "nested" in kinds(r, "heading")[0]["content"])
    run("abstract", r"\begin{abstract}Summary.\end{abstract}\section{A}Body",
        lambda r: kinds(r, "text_block")[0]["parent_id"] == kinds(r, "abstract")[0]["node_id"])
    run("acknowledgements", r"\section*{Acknowledgements}Thanks.",
        lambda r: "Thanks" in content(r) and kinds(r, "heading")[0]["metadata"]["starred"])
    run("reference_then_appendix", r"\section{A}Body\section*{References}DROP\subsection{More refs}DROP2\appendix\section{Extra}KEEP",
        lambda r: "DROP" not in content(r) and "KEEP" in content(r) and kinds(r, "heading")[-1]["metadata"]["is_appendix"])
    run("reference_then_section", r"\section{References}DROP\section{Acknowledgements}KEEP",
        lambda r: "DROP" not in content(r) and "KEEP" in content(r))
    run("formatted_references", r"\section*{\textbf{References}}DROP\appendix\section{Extra}KEEP",
        lambda r: "DROP" not in content(r) and "KEEP" in content(r))
    run("bibliography_env", r"Before\begin{thebibliography}{99}\bibitem{x}DROP\end{thebibliography}After",
        lambda r: "DROP" not in content(r) and "Before" in content(r) and "After" in content(r))
    run("bibliography_commands", r"Before\bibliographystyle{plain}\bibliography{refs}\printbibliography[heading=bibintoc]\appendix\section{A}After",
        lambda r: "bibliography" not in content(r) and "After" in content(r))
    run("references_word_in_prose", "Our References field is useful.", lambda r: "References" in content(r))
    run("display_math", r"A $$x=1.$$ B \[y=2\] C \begin{align}a&=b\\c&=d\end{align}",
        lambda r: len(kinds(r, "formula")) == 3)
    run("inline_math_protected", r"Text $\section{NOT A HEADING}$ and \(x.y\).",
        lambda r: not kinds(r, "heading") and not kinds(r, "formula") and r"\section" in content(r))
    run("figure_table_nested", r"\begin{figure}\begin{center}\caption{A {nested} caption}\end{center}\end{figure}\begin{table}\begin{tabular}{cc}a&b\end{tabular}\end{table}",
        lambda r: len(kinds(r, "figure")) == 2 and not warnings(r))
    r = run("nested_list", r"\begin{itemize}\item Outer.\begin{itemize}\item Inner.\end{itemize}\item[Label] Second.\end{itemize}",
            lambda r: len(kinds(r, "itemize")) == 2 and len(kinds(r, "item")) == 3)
    assert kinds(r, "itemize")[1]["parent_id"] == kinds(r, "item")[0]["node_id"]
    assert kinds(r, "item")[-1]["metadata"]["label"] == "Label"
    run("list_formula_item_literal", r"\begin{itemize}\item A.\begin{equation}\text{\item Fake}\end{equation}\item B.\end{itemize}",
        lambda r: len(kinds(r, "item")) == 2 and len(kinds(r, "formula")) == 1)
    run("enumerate_description", r"\begin{enumerate}\item A\end{enumerate}\begin{description}\item[X] B\end{description}",
        lambda r: len(kinds(r, "itemize")) == 2)
    run("comment_and_escaped_percent", "% \\section{FAKE}\n5\\% real.\\section{A}Body",
        lambda r: len(kinds(r, "heading")) == 1 and r"5\%" in content(r))
    run("macro_body_not_heading", r"\newcommand{\thing}{\section{FAKE}}Actual.",
        lambda r: not kinds(r, "heading"))
    run("literal_environment", r"\begin{verbatim}\section{FAKE}\end{verbatim}",
        lambda r: not kinds(r, "heading") and any(i["code"] == "literal_environment" for i in warnings(r)))
    run("verb_inline", r"Example \verb|\section{FAKE}| is literal.", lambda r: not kinds(r, "heading"))
    run("unclosed_env", r"Before\begin{figure}KEEP", lambda r: "KEEP" in content(r) and bool(warnings(r)))
    run("unclosed_formula", r"Before $$KEEP", lambda r: "KEEP" in content(r) and bool(warnings(r)))
    run("unknown_environment", r"\begin{custom}KEEP\end{custom}",
        lambda r: "KEEP" in content(r) and any(i["code"] == "unknown_environment" for i in warnings(r)))
    run("repeated_occurrences", r"\section{A}Same.\section{B}Same.",
        lambda r: len(kinds(r, "text_block")) == 2 and len({n["node_id"] for n in kinds(r, "text_block")}) == 2)
    run("empty_document", "", lambda r: not r["automatic_checks_passed"])
    run("unresolved_include", r"A\input{missing}", lambda r: any(i["code"] == "unresolved_include" for i in warnings(r)))
    run("appendices_environment", r"\begin{appendices}\section{Extra}KEEP\end{appendices}",
        lambda r: kinds(r, "heading")[0]["metadata"]["is_appendix"])
    # These tiny adapters are ONLY test fixtures; never used for paper parsing.
    run("member4_hooks_contract", "First. Second.",
        lambda r: len(kinds(r, "sentence")) == 2 and r["pipeline_candidate"],
        macro_expander=lambda pre, body: body,
        text_normalizer=lambda text: text,
        sentence_splitter=lambda text: [s.strip() + "." for s in text.split(".") if s.strip()],
        macro_processing_required=True)
    run("macro_hook_missing", "Text.", lambda r: not r["pipeline_candidate"] and any(i["code"] == "macro_hook_missing" for i in r["issues"]), macro_processing_required=True)
    run("malformed_inner_environment", r"\begin{table}\begin{tabular}{c}a\end{table}After.",
        lambda r: len(kinds(r, "figure")) == 1 and "After" in content(r))
    run("loose_bibitem_excluded",
        "Body.\n\n\\bibitem{a} A. Author, Title, 2020.\n\n\\bibitem{b} B. Author, Other, 2021.",
        lambda r: "Author" not in content(r) and "Body" in content(r))
    invalid = [{"node_id": "n0", "parent_id": None, "kind": "document", "order": 0},
               {"node_id": "n1", "parent_id": "missing", "kind": "sentence", "order": 0}]
    assert not validate_tree(invalid)["valid"]
    passed.append("validator_rejects_missing_parent")
    cyclic = [{"node_id": "n0", "parent_id": None, "kind": "document", "order": 0},
              {"node_id": "n1", "parent_id": "n2", "kind": "heading", "order": 0},
              {"node_id": "n2", "parent_id": "n1", "kind": "heading", "order": 0}]
    assert not validate_tree(cyclic)["valid"]
    passed.append("validator_rejects_cycle")
    try:
        build_tree_report("Text.", sentence_splitter=lambda text: "not a list")
    except ValueError:
        passed.append("invalid_splitter_rejected")
    else:
        raise AssertionError("Invalid splitter was accepted")
    return {"passed": len(passed), "failed": 0, "cases": passed}


TASK3_TEST_REPORT = run_build_tree_tests()
print(f"Task 3 tests: {TASK3_TEST_REPORT["passed"]} passed, {TASK3_TEST_REPORT["failed"]} failed")


def run_pipeline_tree_tests():
    """Check the tree shape and representative content consumed by notebook 13."""
    cases = []

    def export(label, text):
        def sentence_splitter(paragraph):
            return re.findall(r"[^.]+[.]|[^.]+$", paragraph)
        tree = build_tree(text, sentence_splitter=sentence_splitter)
        allowed = {"document", "chapter", "section", "subsection", "subsubsection",
                   "paragraph", "abstract", "itemize", "item", "sentence", "formula", "figure"}

        def walk(node):
            assert node.keys() == {"kind", "content", "children"}, node
            assert node["kind"] in allowed and isinstance(node["content"], str), node
            if node["kind"] in {"sentence", "formula", "figure"}:
                assert not node["children"], node
            for child in node["children"]:
                walk(child)

        walk(tree)
        cases.append(label)
        return tree

    tree = export("export_headings_and_sentences",
                  r"\section{Intro}First. Second.\subsection{Details}Third.")
    section = tree["children"][0]
    assert section["kind"] == "section"
    assert [n["content"] for n in section["children"]] == ["First.", "Second.", "Details"]
    assert section["children"][2]["kind"] == "subsection"

    tree = export("export_full_document_title_and_abstract",
                  r"\documentclass{article}\title[Short]{Full Title}\abstract{Overview.}"
                  r"\begin{document}\section{Intro}Body.\end{document}Trailing.")
    assert tree["content"] == "Full Title"
    assert [n["kind"] for n in tree["children"]] == ["abstract", "section"]
    assert tree["children"][0]["children"][0]["content"] == "Overview."

    tree = export("export_formula_and_figure",
                  r"\begin{align*}x=y\end{align*}\begin{wrapfigure}{r}{3cm}\caption{Example}\end{wrapfigure}")
    assert [n["kind"] for n in tree["children"]] == ["formula", "figure"]

    tree = export("export_list_labels_and_transparent_environments",
                  r"\begin{quote}\begin{enumerate}\item[Label]Item.\end{enumerate}\end{quote}")
    item = tree["children"][0]["children"][0]
    assert item["kind"] == "item" and item["content"] == "Label"
    assert item["children"][0]["content"] == "Item."

    tree = export("export_appendix_without_references",
                  r"\section{References}Omit.\appendix\section{Appendix}Keep.")
    assert [n["content"] for n in tree["children"]] == ["Appendix"]
    assert tree["children"][0]["children"][0]["content"] == "Keep."

    tree = export("export_heading_without_label_and_formula_in_footnote",
                  r"\section{Heading\label{sec:heading}}Main text.\footnote{Note before."
                  r"\begin{equation}x=y\end{equation}Note after.}End.")
    section = tree["children"][0]
    assert section["content"] == "Heading"
    assert [n["kind"] for n in section["children"]] == ["sentence", "sentence", "formula", "sentence", "sentence"]
    assert section["children"][2]["content"] == r"\begin{equation}x=y\end{equation}"

    tree = export("export_literal_block_as_raw_figure",
                  "\\begin{lstlisting}\nprint('code')\n\\end{lstlisting}")
    assert tree["children"][0]["kind"] == "figure" and "print" in tree["children"][0]["content"]

    return {"passed": len(cases), "failed": 0, "cases": cases}


PIPELINE_TREE_TEST_REPORT = run_pipeline_tree_tests()
print(PIPELINE_TREE_TEST_REPORT)


## 12. Dùng nguyên hàm từ notebook 11

Trên Colab, upload thêm **11_normalize.ipynb nguyên bản** vào /content (cùng nơi với notebook 10).
Local/repo: đặt file trong cùng thư mục hoặc notebooks/, hoặc sửa NORMALIZE_NOTEBOOK_PATH bên dưới.

Chỉ nạp function definitions/imports/constants của các cell chứa expand_macros, clean_latex,
normalize_math, split_sentences. Không chạy config, pipeline toàn corpus, ghi normalized.txt hoặc biểu đồ.
Hàm gốc không bị sửa, thay thế hay monkey-patch. Adapter Member 2 chỉ truyền preamble/body đã gather
vào đúng tham số và gọi các hàm gốc. SHA-256 được ghi để nhóm biết đang review phiên bản notebook 11 nào.

Chưa có notebook 11: vẫn xem được cây trung gian text_block; báo cáo ghi rõ dependency chưa nạp.
Các lỗi đã biết của notebook 11 là giới hạn tích hợp để Member 4 xem xét, không được tự vá tại notebook 10.


In [ ]:
"""Read original normalization function definitions without running notebook pipeline."""
import ast
import hashlib
import json
from pathlib import Path
from types import SimpleNamespace


def load_member11_functions(notebook_path):
    path = Path(notebook_path)
    raw = path.read_bytes()
    notebook = json.loads(raw)
    public = {'expand_macros', 'clean_latex', 'normalize_math', 'split_sentences'}
    namespace = {'__name__': 'member11_original_functions'}
    selected_cells = []
    for index, cell in enumerate(notebook.get('cells', [])):
        if cell.get('cell_type') != 'code':
            continue
        source = cell.get('source', '')
        source = ''.join(source) if isinstance(source, list) else source
        try:
            parsed = ast.parse(source)
        except SyntaxError:
            # Config/magic cells are not needed by the normalization functions.
            continue
        names = {node.name for node in parsed.body if isinstance(node, ast.FunctionDef)}
        selected = bool(names & public)
        statements = []
        for node in parsed.body:
            if isinstance(node, ast.Import):
                allowed = {'re', 'typing', 'unicodedata'}
                imports = [alias for alias in node.names if alias.name in allowed]
                if selected and len(imports) != len(node.names):
                    raise ImportError(f'Unexpected dependency in normalization cell {index}; review before loading')
                if imports:
                    statements.append(ast.copy_location(ast.Import(names=imports), node))
            elif isinstance(node, ast.ImportFrom):
                if node.module in {'typing', '__future__'}:
                    statements.append(node)
                elif selected:
                    raise ImportError(f'Unexpected dependency in normalization cell {index}; review before loading')
            elif selected and isinstance(node, (ast.FunctionDef, ast.Assign, ast.AnnAssign)):
                statements.append(node)
        # Safe imports can live in a separate notebook cell. Function bodies and
        # constants in the selected definition cells are compiled unchanged.
        if statements:
            module = ast.fix_missing_locations(ast.Module(body=statements, type_ignores=[]))
            exec(compile(module, f'{path.name}:cell_{index}', 'exec'), namespace)
        if selected:
            selected_cells.append(index)
    needed = public | {'_parse_macros'}
    missing = sorted(name for name in needed if not callable(namespace.get(name)))
    if missing:
        raise ValueError(f'Notebook 11 is missing required functions: {missing}')
    return SimpleNamespace(**{name: namespace[name] for name in needed}), {
        'file': str(path), 'sha256': hashlib.sha256(raw).hexdigest(),
        'loaded_cells': selected_cells, 'function_bodies_modified': False,
        'full_notebook_executed': False,
    }


def apply_original_member11_nodes(tree_result, member11):
    if member11 is None:
        return tree_result
    for node in tree_result['nodes']:
        if node['kind'] in {'heading', 'formula', 'figure'}:
            node['content'] = member11.normalize_math(member11.clean_latex(node['content']))
    # Detect unsupported remaining controls; Member 2 records them, never evaluates them.
    import re
    controls = re.findall(r'\\(?:ifdim|ifnum|ifx|ifdefined|iftrue|iffalse|ifcase|else|fi)\b', tree_result['source_text'])
    if controls:
        tree_result['issues'].append({
            'code': 'member11_unresolved_controls', 'start': 0, 'severity': 'warning',
            'message': 'Original notebook 11 left TeX controls in the expanded body: ' + ', '.join(sorted(set(controls))),
        })
        tree_result['automatic_checks_passed'] = False
        tree_result['pipeline_candidate'] = False
    return tree_result


NORMALIZE_NOTEBOOK_PATH = None  # Ví dụ Path("/content/11_normalize.ipynb")
MEMBER11 = None
MEMBER11_SOURCE = None
MACRO_EXPANDER = None
TEXT_NORMALIZER = None
SENTENCE_SPLITTER = None

if NORMALIZE_NOTEBOOK_PATH is None:
    candidates = (Path("11_normalize.ipynb"), Path("notebooks/11_normalize.ipynb"))
    NORMALIZE_NOTEBOOK_PATH = next((p for p in candidates if p.is_file()), None)

if NORMALIZE_NOTEBOOK_PATH is None:
    print("Chưa có notebook 11: Task 3 sẽ tạo cây trung gian với text_block.")
else:
    MEMBER11, MEMBER11_SOURCE = load_member11_functions(NORMALIZE_NOTEBOOK_PATH)
    print("Member 11 source:", MEMBER11_SOURCE)

    def _member2_macro_adapter(pre, body):
        macros = MEMBER11._parse_macros(pre)
        expanded = MEMBER11.expand_macros(body, macros=macros)
        return MEMBER11.normalize_math(expanded)

    def _member2_text_adapter(text):
        return MEMBER11.normalize_math(MEMBER11.clean_latex(text))

    MACRO_EXPANDER = _member2_macro_adapter
    TEXT_NORMALIZER = _member2_text_adapter
    SENTENCE_SPLITTER = MEMBER11.split_sentences


## 13. Parse version đang chọn

Chỉ dùng output gather status=ok. Nếu gather partial/error, xử lý cảnh báo trước.
Những thông báo hook_missing là ghi chú tích hợp; warning thật cần xem trước khi bàn giao.

In [ ]:
tree_result = None

if gather_result is None or gather_result["status"] != "ok":
    print("Chưa chạy Task 3: cần gather status=ok trước.")
else:
    tree_result = build_tree_report(
        gather_result["body"],
        preamble=gather_result["preamble"],
        macro_expander=MACRO_EXPANDER,
        text_normalizer=TEXT_NORMALIZER,
        sentence_splitter=SENTENCE_SPLITTER,
        macro_processing_required=gather_result["needs_macro_processing"],
    )
    print("Paper/version:", PAPER_ID, VERSION)
    print("Structure valid:", tree_result["structure_valid"])
    print("Automatic structural checks:", tree_result["automatic_checks_passed"])
    print("Sentence leaves complete:", tree_result["sentence_leaves_complete"])
    print("Member 4 hooks integrated:", tree_result["member4_hooks_integrated"])
    print("Nodes:", len(tree_result["nodes"]))
    print("Counts:", tree_result["counts"])
    print("Excluded regions:", len(tree_result["excluded"]))
    print("\nValidation errors:", tree_result["validation"]["errors"] or "(none)")
    print("\nIssues (tối đa 30; đầy đủ trong tree_result['issues']):")
    for item in tree_result["issues"][:30]:
        print(f"[{item['severity']}] {item['code']} at {item['start']}: {item['message']}")
    if not tree_result["issues"]:
        print("(none)")

## 14. Xem cây và Appendix

Xem preview, rồi kiểm tra thêm heading cuối để không bỏ sót Appendix ở xa đầu paper.
Dữ liệu đầy đủ nằm trong tree_result['nodes']; preview chỉ giới hạn phần hiển thị.

In [ ]:
if tree_result is not None:
    print(preview_tree(tree_result, max_nodes=60))
    heading_rows = [
        {
            "node_id": n["node_id"], "parent_id": n["parent_id"],
            "title": n["content"], "command": n["metadata"]["command"],
            "appendix": n["metadata"]["is_appendix"],
        }
        for n in tree_result["nodes"] if n["kind"] == "heading"
    ]
    print("\n15 heading cuối:")
    try:
        import pandas as pd
        from IPython.display import display
    except ImportError:
        for row in heading_rows[-15:]:
            print(row)
    else:
        display(pd.DataFrame(heading_rows[-15:]))

## 15. Báo cáo để nhóm review

Push review không có nghĩa là đã kiểm định hoàn chỉnh trên corpus hoặc đã hết lỗi notebook 11.
Báo cáo này ghi nguồn notebook 11, kết quả cây trung gian/tích hợp và các cảnh báo còn tồn tại.
Không có checkbox bắt buộc chờ Member 1/4 hoàn tất công việc trước khi push.

Giới hạn của dependency cần nhóm lưu ý (không vá ở đây): macro lồng nhau/định nghĩa lại,
URL cuối câu, nested command arguments, math chứa dấu gạch và điều kiện TeX chưa thực thi.
Các test cấu trúc pass không chứng minh nội dung chuẩn hóa của notebook 11 đã đúng.


In [ ]:
review_report = {
    "purpose": "peer_review_member2",
    "paper_id": PAPER_ID,
    "version": VERSION,
    "root": None if gather_result is None else gather_result["root_path"],
    "gather_status": None if gather_result is None else gather_result["status"],
    "member2_tree_tests": TASK3_TEST_REPORT,
    "pipeline_tree_tests": PIPELINE_TREE_TEST_REPORT,
    "member11_source": MEMBER11_SOURCE,
    "normalization_functions_modified": False,
    "structure_valid": None if tree_result is None else tree_result["structure_valid"],
    "counts": {} if tree_result is None else tree_result["counts"],
    "sentence_leaves_complete": False if tree_result is None else tree_result["sentence_leaves_complete"],
    "issues": [] if tree_result is None else tree_result["issues"],
    "full_corpus_validated": False,
    "note": "For code review; this report does not certify final pipeline correctness.",
}
import json
print(json.dumps(review_report, ensure_ascii=False, indent=2))


## Nội dung push lên nhánh feat/latex-tree-parser

- notebooks/10_parse_tree.ipynb: code Member 2, adapter gọi nguyên hàm notebook 11, test và diagnostics.
- Nếu nhóm dùng module: src/parse_tree.py (find_root/gather/build_tree) có thể đi cùng.
- Không đưa normalize_member4.py, normalization_comparison.ipynb hay các bản vá A/B vào PR Member 2.
- Không sửa notebook 11. Ghi PR description rằng phụ thuộc notebook 11 nguyên bản và các giới hạn đã biết.
- Xóa output dài/đường dẫn cá nhân trước khi commit. Attach review_report nếu có lỗi cần nhóm xem.


## 16. Regression tests sau feedback

Chạy cell này sau khi đã định nghĩa cả find_root, gather và build_tree.
Kỳ vọng 26 passed, 0 failed. Đây là fixtures tái hiện các lớp lỗi, không phải kết quả
chạy lại 291 version của nhóm. Không tự kết luận đã hết 7 version lệch hoặc 2 root khác.
Tài liệu chính thức về import/subfiles:
- https://ctan.org/pkg/import
- https://ctan.org/pkg/subfiles


In [ ]:
def run_feedback_regressions():
    """Reproduce progress-review failure classes, not the unseen 291-version audit."""
    import tempfile
    from pathlib import Path
    passed = []
    # Colab /content is writable; locally put fixtures under current workspace.
    with tempfile.TemporaryDirectory(dir=Path.cwd()) as temp:
        base = Path(temp).resolve()
        assert base.is_relative_to(Path.cwd().resolve())

        def check(label, files, expected_root, expected_text, absent=()):
            folder = base / label
            folder.mkdir()
            for relative, text in files.items():
                path = folder / relative
                path.parent.mkdir(parents=True, exist_ok=True)
                path.write_text(text, encoding="utf-8")
            root = find_root(folder)
            assert root["root_path"] == expected_root, (label, root)
            output = gather(folder, root["root_path"])
            assert output["status"] == "ok", (label, output["warnings"])
            for marker in expected_text:
                assert marker in output["body"], (label, marker, output["body"])
            for marker in absent:
                assert marker not in output["body"], (label, marker)
            tree = build_tree_report(output["body"], preamble=output["preamble"])
            assert tree["structure_valid"], (label, tree["validation"])
            passed.append(label)
            return root, output, tree

        doc = r"\documentclass{article}\begin{document}BODY\end{document}"
        root, _, _ = check("class_in_preamble", {
            "main.tex": r"\input{preamble}\begin{document}BODY\end{document}",
            "preamble.tex": r"\documentclass{article}\newcommand{\x}{x}",
        }, "main.tex", ["BODY"])
        assert root["candidates"][0]["documentclass_source"] == "preamble.tex"
        check("class_and_begin_in_include", {
            "main.tex": r"\input{setup}\input{Sections/1.Introduction}\end{document}",
            "setup.tex": r"\documentclass{article}\begin{document}",
            "Sections/1.Introduction.tex": r"\section{Intro}INTRO",
        }, "main.tex", ["INTRO"])
        check("dots_in_basename", {
            "main.tex": doc.replace("BODY", r"\input{Sections/1.Introduction}"),
            "Sections/1.Introduction.tex": "DOTS_WORK",
        }, "main.tex", ["DOTS_WORK"])
        check("input_with_explicit_extension", {
            "main.tex": doc.replace("BODY", r"\input{Sections/1.Introduction.tex}"),
            "Sections/1.Introduction.tex": "EXPLICIT",
        }, "main.tex", ["EXPLICIT"])
        check("subfile_body_only", {
            "main.tex": doc.replace("BODY", r"\subfile{Sections/part}"),
            "Sections/part.tex": r"\documentclass[../main.tex]{subfiles}\newcommand{\PREAMBLEONLY}{bad}\begin{document}SUBBODY\input{detail}\end{document}",
            "Sections/detail.tex": "LOCAL_DETAIL",
        }, "main.tex", ["SUBBODY", "LOCAL_DETAIL"], ["PREAMBLEONLY", "documentclass", "begin{document}"])
        check("import_directory_context", {
            "main.tex": doc.replace("BODY", r"\import{Sections/}{part}"),
            "Sections/part.tex": r"PART\input{detail}",
            "Sections/detail.tex": "IMPORTED_DETAIL",
        }, "main.tex", ["PART", "IMPORTED_DETAIL"])
        check("nested_subimport", {
            "main.tex": doc.replace("BODY", r"\import{Sections/}{part}"),
            "Sections/part.tex": r"PART\subimport{more/}{detail}",
            "Sections/more/detail.tex": "NESTED",
        }, "main.tex", ["PART", "NESTED"])
        check("inputfrom_alias", {
            "main.tex": doc.replace("BODY", r"\inputfrom{Sections/}{part}"),
            "Sections/part.tex": "ALIAS",
        }, "main.tex", ["ALIAS"])
        _, output, _ = check("repeat_subfile", {
            "main.tex": doc.replace("BODY", r"\subfile{part}\subfile{part}"),
            "part.tex": r"\documentclass[main.tex]{subfiles}\begin{document}TWICE\end{document}",
        }, "main.tex", ["TWICE"])
        assert output["body"].count("TWICE") == 2
        _, output, _ = check("unused_file_and_macro_definition", {
            "main.tex": doc.replace("BODY", r"\newcommand{\unused}{\input{unused}}USED"),
            "unused.tex": "NOT_COMPILED",
        }, "main.tex", ["USED"], ["NOT_COMPILED"])
        assert len(output["files_read"]) == 1
        check("commented_extended_commands", {
            "main.tex": "\\documentclass{article}\n% \\import{bad/}{missing}\n\\begin{document}BODY% \\subfile{missing}\n\\end{document}",
        }, "main.tex", ["BODY"])
        check("import_stays_within_its_directory", {
            "main.tex": doc.replace("BODY", r"\import{Sections/}{part}"),
            "Sections/part.tex": "CORRECT_IMPORT",
            "part.tex": "WRONG_ROOT_LEVEL",
        }, "main.tex", ["CORRECT_IMPORT"], ["WRONG_ROOT_LEVEL"])

        root, output, _ = check("wrapper_preserves_macro", {
            "wrapper.tex": r"\newcommand{\fromwrapper}{Wrapper text.}\input{main}",
            "main.tex": doc.replace("BODY", r"\fromwrapper"),
        }, "wrapper.tex", [r"\fromwrapper"])
        assert r"\newcommand{\fromwrapper}" in output["preamble"]
        assert next(c for c in root["candidates"] if c["path"] == "main.tex")["excluded"]

        for ext in ("def", "ltx"):
            path = f"preamble.{ext}"
            root, _, _ = check(f"class_in_{ext}", {
                "main.tex": "\\input{" + path + "}" + r"\begin{document}BODY\end{document}",
                path: r"\documentclass{article}",
            }, "main.tex", ["BODY"])
            assert root["candidates"][0]["documentclass_source"] == path

        check("macro_document_markers_are_inert", {
            "main.tex": r"\documentclass{article}\newcommand{\unused}{\begin{document}FAKE\end{document}}"
                        r"\begin{document}BODY\end{document}",
        }, "main.tex", ["BODY"], ["FAKE"])
        check("fake_documentclass_does_not_choose_root", {
            "main.tex": r"\newcommand{\unused}{\documentclass{bogus}}\input{real.def}\begin{document}BODY\end{document}",
            "real.def": r"\documentclass{article}",
        }, "main.tex", ["BODY"])
        _, output, _ = check("ignore_includes_after_document", {
            "main.tex": doc + "\n" + r"\input{missing}",
        }, "main.tex", ["BODY"])
        assert not output["include_events"] and r"\input{missing}" in output["trailing_text"]
        _, output, _ = check("stop_document_inside_include", {
            "main.tex": r"\documentclass{article}\begin{document}\input{body}\input{missing_outer}",
            "body.tex": r"BODY\end{document}\input{missing_inner}",
        }, "main.tex", ["BODY"])
        assert [e["target"] for e in output["include_events"]] == ["body.tex"]
        assert "missing_inner" in output["trailing_text"] and "missing_outer" in output["trailing_text"]
        _, output, _ = check("stop_file_at_endinput_line", {
            "main.tex": r"\input{setup}\begin{document}BODY\end{document}",
            "setup.tex": r"\documentclass{article}" + "\n" + r"\endinput % stop after this line"
                         + "\n" + r"\input{missing}",
        }, "main.tex", ["BODY"])
        assert [e["target"] for e in output["include_events"]] == ["setup.tex"]
        assert "endinput" not in output["expanded_text"]
        _, output, _ = check("nested_import_prefers_active_context", {
            "main.tex": doc.replace("BODY", r"\import{chap/}{part}"),
            "chap/part.tex": r"\input{sub/part2}",
            "chap/sub/part2.tex": r"\input{detail}",
            "chap/detail.tex": "RIGHT",
            "chap/sub/detail.tex": "WRONG",
        }, "main.tex", ["RIGHT"], ["WRONG"])
        assert output["include_events"][-1]["target"] == "chap/detail.tex"
        assert any(item["code"] == "include_precedence" for item in output["issues"])

        check("input_inside_braces", {
            "main.tex": doc.replace("BODY", r"\resizebox{\textwidth}{!}{\input{table}}"
                                             r"\IfFileExists{num.tex}{\input{num}}{}"),
            "table.tex": "TABLE_ROWS", "num.tex": "NUMBERS",
        }, "main.tex", ["TABLE_ROWS", "NUMBERS"])
        _, output, _ = check("comment_environment_is_not_gathered", {
            "main.tex": doc.replace("BODY", r"VISIBLE\begin{comment}\input{missing}HIDDEN\end{comment}"),
        }, "main.tex", ["VISIBLE"], ["HIDDEN"])
        assert not output["include_events"]

        for label, content, expect, code in (
            ("active_missing_source_is_an_error", r"\input{missing}", "error", "missing_file"),
            ("missing_bibliography_is_reported", r"\input{refs.bbl}", "partial", "missing_file"),
            ("active_cycle_is_an_error", r"\input{main}", "error", "cycle"),
        ):
            folder = base / label
            folder.mkdir()
            (folder / "main.tex").write_text(doc.replace("BODY", content), encoding="utf-8")
            output = gather(folder, "main.tex")
            assert output["status"] == expect and any(item["code"] == code for item in output["issues"]), output
            passed.append(label)
    return {"passed": len(passed), "failed": 0, "cases": passed}


FEEDBACK_TEST_REPORT = run_feedback_regressions()
print(FEEDBACK_TEST_REPORT)

## 17. Audit các version của nhóm (tùy chọn)

Mặc định tắt. Đặt RUN_MEMBER2_AUDIT=True để chạy; SELECTED_VERSIONS=None quét mọi paper/vN.
Để thử trước, đặt [("2603-25579", "v2")]. Không sửa file Drive.

Muốn so sánh với fallback cần baseline THẬT của nhóm, cùng cách đo text length.
Không có baseline thì root_differs/over_10_percent để None, không tự ghi nhận đã hết lỗi.
Đây là audit cây trung gian, chưa thay benchmark cuối hoặc bước chuẩn hóa/tách câu Member 4.


In [ ]:
def audit_member2(raw_dir, *, selected=None, baseline=None):
    """Read-only audit, one record per version. No fallback text is fabricated.

    selected: optional list[(paper_id, 'v1')]. None scans paper/vN directories.
    baseline: optional mapping {(paper_id, version):
        {'root': 'main.tex', 'expanded_char_count': int}}.
    Baseline lengths MUST use the same definition: source after gathering/comments,
    before body extraction. Different cleanup rules invalidate this comparison.
    """
    raw_dir = Path(raw_dir)
    baseline = baseline or {}
    if selected is None:
        selected = sorted(
            (paper.name, version.name)
            for paper in raw_dir.iterdir() if paper.is_dir()
            for version in paper.iterdir()
            if version.is_dir() and re.fullmatch(r"v\d+", version.name)
        )
    rows = []
    for paper, version in selected:
        row = {"paper_id": paper, "version": version, "root": None,
               "root_ambiguous": False, "gather_status": None,
               "expanded_char_count": None, "tree_structure_valid": None,
               "tree_warnings": None, "root_differs": None,
               "relative_char_difference": None, "over_10_percent": None}
        try:
            folder = raw_dir / paper / version
            root = find_root(folder)
            row.update(root=root["root_path"], root_ambiguous=root["ambiguous"])
            if root["status"] != "ok":
                row["gather_status"] = "empty_source" if root["tex_file_count"] == 0 else "no_root"
                row["issues"] = root["warnings"]
            else:
                gathered = gather(folder, root["root_path"])
                row.update(gather_status=gathered["status"],
                           expanded_char_count=gathered["expanded_char_count"],
                           include_count=len(gathered["include_events"]),
                           files_read=len(gathered["files_read"]), issues=gathered["warnings"])
                if gathered["status"] == "ok":
                    tree = build_tree_report(gathered["body"], preamble=gathered["preamble"],
                                      macro_processing_required=gathered["needs_macro_processing"])
                    row.update(tree_structure_valid=tree["structure_valid"],
                               tree_warnings=sum(i["severity"] == "warning" for i in tree["issues"]),
                               node_counts=tree["counts"])
            old = baseline.get((paper, version))
            if old is not None:
                if "root" in old:
                    row["root_differs"] = row["root"] != old["root"]
                old_count = old.get("expanded_char_count")
                if old_count is not None and row["expanded_char_count"] is not None:
                    delta = abs(row["expanded_char_count"] - old_count) / max(old_count, 1)
                    row.update(relative_char_difference=delta, over_10_percent=delta > 0.10)
        except Exception as exc:
            row["gather_status"] = "error"
            row["issues"] = [f"{type(exc).__name__}: {exc}"]
        rows.append(row)
    return rows


RUN_MEMBER2_AUDIT = False
SELECTED_VERSIONS = [("2603-25579", "v2")]
REVIEW_BASELINE = {}
# Ví dụ schema baseline, chỉ điền bằng số liệu audit thật:
# REVIEW_BASELINE[("paper_id", "v1")] = {"root": "main.tex", "expanded_char_count": 12345}

if RUN_MEMBER2_AUDIT:
    audit_rows = audit_member2(RAW_DIR, selected=SELECTED_VERSIONS, baseline=REVIEW_BASELINE)
    try:
        import pandas as pd
        from IPython.display import display
    except ImportError:
        for row in audit_rows:
            print(row)
    else:
        display(pd.DataFrame(audit_rows))
else:
    print("Batch audit is off. Enable only after checking the selected source paths.")
